# Bengali EVC v3 - Emotion Voice Conversion with F0 Supervision
## Google Colab A100 Edition - NSF-HiFiGAN vocoder

**Pipeline:** content/aux encoders + ProsodyHead predict the target-emotion mel **and** its F0 contour; an **NSF-HiFiGAN** neural source-filter vocoder (fine-tuned on raw SUBESCO) renders that mel+F0 into natural Bangla speech so the injected emotion is actually audible.

**Vocoder:** NSF-HiFiGAN only (Griffin-Lim kept solely as an emergency fallback if the vocoder is untrained). Tuned for a single **Colab A100** (TF32 + bf16 AMP).

**Run on:** Google Colab with A100/T4 GPU runtime.

## 1 · Colab Setup — Mount Drive, Install Dependencies, Download Dataset

In [ ]:
# ============================================================
# BLOCK 1 — Google Colab Setup: Drive + Dependencies + Dataset
# ============================================================

# --- Mount Google Drive for persistent output ---
from google.colab import drive
drive.mount('/content/drive')

# --- Install dependencies ---
import subprocess, sys
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("librosa", "soundfile", "kaggle")

# --- Setup Kaggle API for dataset download ---
# You need to upload your kaggle.json or set env vars
import os
from pathlib import Path

# Option A: Upload kaggle.json manually to /content/
# Option B: Set env vars (uncomment and fill):
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

# If kaggle.json exists in /content/, copy it
if Path('/content/kaggle.json').exists():
    import shutil
    shutil.copy('/content/kaggle.json', kaggle_dir / 'kaggle.json')
    os.chmod(str(kaggle_dir / 'kaggle.json'), 0o600)
    print("Kaggle credentials configured from /content/kaggle.json")
elif (kaggle_dir / 'kaggle.json').exists():
    print("Kaggle credentials already configured")
else:
    print("WARNING: No kaggle.json found!")
    print("  Upload kaggle.json to /content/ or set KAGGLE_USERNAME/KAGGLE_KEY env vars")
    print("  Get yours from: https://www.kaggle.com/settings -> API -> Create New Token")

print("\nSetup complete. Drive mounted at /content/drive/")


## 2 · Download Datasets from Kaggle

In [ ]:
# ============================================================
# BLOCK 2 — Download feature dataset + checkpoint from Kaggle
# ============================================================

import subprocess
from pathlib import Path

KAGGLE_DATASET_SLUG = "yousufasgormumin57/4-emo-dataset"
KAGGLE_CHECKPOINT_SLUG = "yousufasgormumin57/checkpoint-a-i-r"

DATA_DOWNLOAD_DIR = Path("/content/kaggle_data")
CKPT_DOWNLOAD_DIR = Path("/content/kaggle_checkpoint")

DATA_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Download feature dataset
if not list(DATA_DOWNLOAD_DIR.rglob("metadata.csv")):
    print(f"Downloading dataset: {KAGGLE_DATASET_SLUG}")
    subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET_SLUG,
                    "-p", str(DATA_DOWNLOAD_DIR), "--unzip"], check=True)
    print("Dataset downloaded!")
else:
    print("Dataset already present.")

# Download checkpoint
if not list(CKPT_DOWNLOAD_DIR.rglob("*.pt")):
    print(f"Downloading checkpoint: {KAGGLE_CHECKPOINT_SLUG}")
    subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_CHECKPOINT_SLUG,
                    "-p", str(CKPT_DOWNLOAD_DIR), "--unzip"], check=True)
    print("Checkpoint downloaded!")
else:
    print("Checkpoint already present.")

# Show what we got
print("\nDataset contents:")
for f in sorted(DATA_DOWNLOAD_DIR.rglob("*"))[:20]:
    if f.is_file():
        print(f"  {f.relative_to(DATA_DOWNLOAD_DIR)}")

print("\nCheckpoint contents:")
for f in sorted(CKPT_DOWNLOAD_DIR.rglob("*.pt")):
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")


## 3 · Imports, Reproducibility, Configuration

In [ ]:
# ============================================================
# BLOCK 3 — Imports + Config (Colab A100 + Drive output)
# ============================================================

import os, json, math, random, shutil, zipfile, copy, time, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import librosa.display
import librosa.sequence
import soundfile as sf

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from scipy import signal

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from IPython.display import Audio, display

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
# --- A100: enable TF32 (large free speedup for fp32 matmul/conv) ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f"  GPU: {prop.name} | {prop.total_mem / 1024**3:.1f} GB")

# ─── Paths (Google Colab + Drive) ────────────────────────────────────────────
KAGGLE_INPUT_ROOT = Path("/content/kaggle_data")
CHECKPOINT_INPUT_DIR = Path("/content/kaggle_checkpoint")

# Output goes to Drive (persists after runtime disconnects)
DRIVE_ROOT = Path("/content/drive/MyDrive/EVC_Output")
OUT_DIR    = DRIVE_ROOT / "evc_v3_output"
CKPT_DIR   = OUT_DIR / "checkpoints"
PLOT_DIR   = OUT_DIR / "plots"
AUDIO_DIR  = OUT_DIR / "audio"
CACHE_DIR  = Path("/content/dtw_cache")  # cache on local SSD (faster)
EXTRACT_DIR = Path("/content/features_extracted")

for p in [OUT_DIR, CKPT_DIR, PLOT_DIR, AUDIO_DIR, CACHE_DIR, EXTRACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CFG = {
    "sample_rate": 16000,
    "n_fft": 2048,
    "hop_length": 512,
    "win_length": 2048,
    "n_mels": 128,
    "fmin": 0.0,
    "fmax": 8000.0,
    "top_db": 25.0,

    "trim_silence": True,
    "trim_use_voiced": True,
    "trim_top_db_margin": 25.0,
    "trim_pad_frames": 5,
    "min_frames_after_trim": 48,
    "edge_zero_window": 5,
    "edge_zero_apply": True,

    "source_emotion": "neutral",
    "target_emotions": ["angry", "happy", "sad"],
    "val_size": 0.10,
    "use_dtw_alignment": True,
    "max_dtw_frames": 420,
    "num_workers": 8,  # A100 runtimes have plenty of CPU cores

    "content_dim": 256,
    "aux_dim": 64,
    "speaker_dim": 128,
    "emotion_dim": 64,
    "prosody_cond_dim": 32,
    "hidden_dim": 256,
    "dropout": 0.10,

    "total_epochs": 300,
    "phase1_epochs": 50,
    "phase2_epochs": 150,
    "phase3_epochs": 100,
    "batch_size": 48,  # A100 can handle larger batches
    "lr_G": 1e-4,
    "lr_D": 5e-5,
    "lr_SER": 1e-4,
    "weight_decay": 1e-5,
    "grad_clip": 5.0,

    "ser_pretrain_epochs": 15,

    "lambda_f0": 8.0,
    "lambda_energy_pred": 4.0,
    "lambda_voiced": 2.0,
    "lambda_prosody": 5.0,
    "lambda_prosody_f0": 6.0,

    "use_f0_transform": True,
    "f0_transform_weight": 0.5,

    "use_online_ser": True,
    "lr_ser_online": 2e-4,
    "online_ser_warmup": 3,
    "use_grl": True,
    "lambda_grl": 1.0,
    "grl_alpha": 1.0,

    "p2_lambda_l1": 3.0,
    "p2_lambda_content": 3.0,
    "p2_lambda_cycle": 1.0,
    "p2_lambda_ser": 4.0,
    "p2_lambda_adv": 0.3,
    "p3_lambda_l1": 2.0,
    "p3_lambda_content": 2.0,
    "p3_lambda_cycle": 0.5,
    "p3_lambda_ser": 6.0,
    "p3_lambda_adv": 0.6,


    "save_every": 25,
    "resume": True,
    "resume_path": None,
    "checkpoint_input_dir": str(CHECKPOINT_INPUT_DIR),
}

print(f"Output dir (Drive): {OUT_DIR}")
print(f"Batch size: {CFG['batch_size']} (optimized for A100)")
print(f"Workers: {CFG['num_workers']}")


## 4 · Locate Feature Dataset

In [ ]:
# ============================================================
# BLOCK 4 — Find processed features (from Kaggle download)
# ============================================================

def find_processed_dataset():
    # Search in the downloaded Kaggle data
    search_roots = [KAGGLE_INPUT_ROOT, EXTRACT_DIR]
    for root in search_roots:
        if not root.exists():
            continue
        candidates = list(root.rglob("metadata.csv"))
        for c in candidates:
            try:
                head = pd.read_csv(c, nrows=2)
                if "mel_path" in head.columns:
                    print(f"Found dataset: {c}")
                    return c, c.parent
            except Exception:
                pass

    # Try extracting a ZIP
    for root in search_roots:
        if not root.exists():
            continue
        for z in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(z) as zf:
                    names = zf.namelist()
                    if any(n.endswith("metadata.csv") for n in names):
                        print(f"Extracting: {z}")
                        zf.extractall(EXTRACT_DIR)
                        meta = list(EXTRACT_DIR.rglob("metadata.csv"))
                        if meta:
                            return meta[0], meta[0].parent
            except Exception:
                continue

    raise FileNotFoundError(
        "Could not find processed dataset (metadata.csv with mel_path column).\n"
        "Make sure the Kaggle dataset was downloaded correctly.")

META_PATH, FEATURE_ROOT = find_processed_dataset()
RESOLVE_ROOTS = [FEATURE_ROOT, EXTRACT_DIR, KAGGLE_INPUT_ROOT]
print(f"Feature root: {FEATURE_ROOT}")


## 5 · Load Metadata + Resolve Paths

In [ ]:
# ============================================================
# BLOCK 5 — Load metadata
# ============================================================

df = pd.read_csv(META_PATH)
df.columns = [c.strip() for c in df.columns]
print(f"Rows: {len(df)}, Columns: {list(df.columns)}")

def pick_col(possible_names, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for name in possible_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    if required:
        raise KeyError(f"Missing column. Tried: {possible_names}")
    return None

COL_EMOTION = pick_col(["emotion"])
COL_LABEL   = pick_col(["label"], required=False)
COL_SPEAKER = pick_col(["speaker"])
COL_SENT    = pick_col(["sentence", "sent"])
COL_TAKE    = pick_col(["take"])
COL_MEL     = pick_col(["mel_path", "mel", "mel_file"])
COL_F0      = pick_col(["f0_path", "f0", "pitch_path"])
COL_ENERGY  = pick_col(["energy_path", "energy", "energy_file"])
COL_VOICED  = pick_col(["voiced_path", "voiced", "uv_path"])
COL_WAV     = pick_col(["wav_path", "audio_path"], required=False)

df[COL_EMOTION] = df[COL_EMOTION].astype(str).str.lower().str.strip()
df[COL_SPEAKER] = df[COL_SPEAKER].astype(str)

print("\nEmotion counts:")
print(df[COL_EMOTION].value_counts().to_string())

def resolve_path(p):
    if pd.isna(p): return None
    p = str(p)
    cand = Path(p)
    if cand.is_absolute() and cand.exists(): return cand
    for root in RESOLVE_ROOTS:
        c = root / p
        if c.exists(): return c
    base = Path(p).name
    for root in RESOLVE_ROOTS:
        hits = list(root.rglob(base))
        if hits: return hits[0]
    return None

# Sanity check
print("\nPath resolution check (first 10):")
for col in [COL_MEL, COL_F0, COL_ENERGY, COL_VOICED]:
    ok = sum(resolve_path(x) is not None for x in df[col].head(10))
    print(f"  {col}: {ok}/10")


## 6 · Feature Loading, Trimming, Normalization

In [ ]:
# ============================================================
# BLOCK 6 — Feature loading utilities
# ============================================================

def ensure_mel_shape(mel):
    mel = np.asarray(mel, dtype=np.float32)
    if mel.ndim != 2: raise ValueError(f"Mel must be 2D, got {mel.shape}")
    if mel.shape[0] == CFG["n_mels"]: return mel
    if mel.shape[1] == CFG["n_mels"]: return mel.T
    raise ValueError(f"No {CFG['n_mels']} dimension in mel: {mel.shape}")

def load_mel_db(path_like):
    path = resolve_path(path_like)
    if path is None: raise FileNotFoundError(path_like)
    mel = np.load(path).astype(np.float32)
    mel = ensure_mel_shape(mel)
    mel = np.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)
    return np.clip(mel, -80.0, 0.0)

def fit_length_1d(x, T):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) == T: return x
    if len(x) > T: return x[:T]
    return np.concatenate([x, np.zeros(T - len(x), dtype=np.float32)])

def fit_length_2d(x, T):
    if x.shape[1] == T: return x
    if x.shape[1] > T: return x[:, :T]
    pad = np.full((x.shape[0], T - x.shape[1]), -80.0, dtype=np.float32)
    return np.concatenate([x, pad], axis=1)

def load_1d_feature(path_like, expected_len=None):
    path = resolve_path(path_like)
    if path is None: raise FileNotFoundError(path_like)
    arr = np.load(path).astype(np.float32).reshape(-1)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if expected_len is not None: arr = fit_length_1d(arr, expected_len)
    return arr

def derive_energy_from_mel_db(mel_db):
    return mel_db.mean(axis=0).astype(np.float32)

def get_active_region(mel_db, voiced=None):
    T = mel_db.shape[1]
    if T <= 1: return 0, T
    frame_energy = derive_energy_from_mel_db(mel_db)
    threshold = float(frame_energy.max()) - CFG["trim_top_db_margin"]
    energy_active = frame_energy > threshold
    if voiced is not None and len(voiced) == T:
        combined = energy_active | (np.asarray(voiced).reshape(-1) > 0.5)
    else:
        combined = energy_active
    idx = np.where(combined)[0]
    if len(idx) < CFG["min_frames_after_trim"]: return 0, T
    pad = CFG["trim_pad_frames"]
    start = max(0, int(idx[0]) - pad)
    end = min(T, int(idx[-1]) + pad + 1)
    return (start, end) if end > start else (0, T)

def zero_silent_edges(mel_db, voiced, energy=None):
    window = CFG["edge_zero_window"]
    T = mel_db.shape[1]
    if T < window * 2 or voiced is None or len(voiced) != T: return mel_db
    if energy is None: energy = derive_energy_from_mel_db(mel_db)
    threshold = float(energy.max()) - CFG["trim_top_db_margin"]
    v = np.asarray(voiced).reshape(-1) > 0.5
    mel_out = mel_db.copy()
    for i in range(window):
        if not v[i] and energy[i] < threshold: mel_out[:, i] = -80.0
    for i in range(T - window, T):
        if not v[i] and energy[i] < threshold: mel_out[:, i] = -80.0
    return mel_out

def trim_feature_bundle(mel_db, f0_hz=None, energy=None, voiced=None):
    if not CFG["trim_silence"]:
        return mel_db, f0_hz, energy, voiced, 0, mel_db.shape[1]
    start, end = get_active_region(mel_db, voiced)
    mel_db = mel_db[:, start:end]
    if f0_hz is not None: f0_hz = f0_hz[start:end]
    if energy is not None: energy = energy[start:end]
    if voiced is not None: voiced = voiced[start:end]
    if CFG["edge_zero_apply"] and voiced is not None:
        mel_db = zero_silent_edges(mel_db, voiced, energy)
    return mel_db, f0_hz, energy, voiced, start, end

def normalize_mel(mel_db):
    return ((np.clip(mel_db, -80.0, 0.0) + 40.0) / 40.0).astype(np.float32)

def denormalize_mel(mel_norm):
    return np.clip(mel_norm * 40.0 - 40.0, -80.0, 0.0).astype(np.float32)

def load_full_features(row):
    mel_db = load_mel_db(row[COL_MEL])
    T = mel_db.shape[1]
    f0_hz = load_1d_feature(row[COL_F0], expected_len=T)
    energy = load_1d_feature(row[COL_ENERGY], expected_len=T)
    voiced = load_1d_feature(row[COL_VOICED], expected_len=T)
    voiced = (voiced > 0.5).astype(np.float32)
    mel_db, f0_hz, energy, voiced, s, e = trim_feature_bundle(
        mel_db, f0_hz, energy, voiced)
    return {"mel_db": mel_db, "f0_hz": f0_hz, "energy": energy, "voiced": voiced}

def compute_stats(df_subset):
    f0_logs, energies, kept = [], [], 0
    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Stats"):
        try:
            feat = load_full_features(row)
        except Exception:
            continue
        f0, e = feat["f0_hz"], feat["energy"]
        voiced_mask = f0 > 0
        if voiced_mask.any():
            f0_logs.append(np.log(np.maximum(f0[voiced_mask], 1e-6)))
        energies.append(e)
        kept += 1
    f0_cat = np.concatenate(f0_logs) if f0_logs else np.array([np.log(150.0)])
    e_cat = np.concatenate(energies) if energies else np.array([-50.0])
    return {"f0_log_mean": float(f0_cat.mean()), "f0_log_std": float(max(f0_cat.std(), 0.01)),
            "energy_mean": float(e_cat.mean()), "energy_std": float(max(e_cat.std(), 0.01)),
            "rows_used": kept}

def normalize_f0(f0_hz, stats):
    f0_hz = np.asarray(f0_hz, dtype=np.float32)
    voiced = f0_hz > 0
    out = np.zeros_like(f0_hz)
    if voiced.any():
        out[voiced] = (np.log(np.maximum(f0_hz[voiced], 1e-6)) - stats["f0_log_mean"]) / (stats["f0_log_std"] + 1e-8)
    return out.astype(np.float32), voiced.astype(np.float32)

def denormalize_f0(f0_norm, voiced, stats):
    f0_norm = np.asarray(f0_norm, dtype=np.float32)
    voiced = np.asarray(voiced) > 0.5
    out = np.zeros_like(f0_norm)
    out[voiced] = np.exp(f0_norm[voiced] * stats["f0_log_std"] + stats["f0_log_mean"])
    return out

def normalize_energy(energy, stats):
    return ((np.asarray(energy, dtype=np.float32) - stats["energy_mean"]) / (stats["energy_std"] + 1e-8)).astype(np.float32)

def denormalize_energy(energy_norm, stats):
    return energy_norm * stats["energy_std"] + stats["energy_mean"]

print("Feature utilities ready.")


## 7 · Build Pairs + Per-Emotion Prosody Statistics

In [ ]:
# ============================================================
# BLOCK 7 — Pairing + per-emotion F0/energy statistics
# ============================================================

source_emo = CFG["source_emotion"]
target_emos = set(CFG["target_emotions"])
df_work = df[df[COL_EMOTION].isin([source_emo] + list(target_emos))].copy().reset_index(drop=True)

emotion_names = sorted(df_work[COL_EMOTION].unique().tolist())
emotion_to_id = {e: i for i, e in enumerate(emotion_names)}
id_to_emotion = {i: e for e, i in emotion_to_id.items()}
speaker_names = sorted(df_work[COL_SPEAKER].unique().tolist())
speaker_to_id = {s: i for i, s in enumerate(speaker_names)}
id_to_speaker = {i: s for s, i in speaker_to_id.items()}

print("Emotions:", emotion_to_id)
print("Speakers:", len(speaker_to_id))

# Build pairs
def make_key(row, include_take=True):
    parts = [str(row[COL_SPEAKER])]
    if COL_SENT: parts.append(str(row[COL_SENT]))
    if include_take and COL_TAKE: parts.append(str(row[COL_TAKE]))
    return "||".join(parts)

neutral_df = df_work[df_work[COL_EMOTION] == source_emo]
target_df = df_work[df_work[COL_EMOTION].isin(target_emos)]

pairs = []
neutral_map = defaultdict(list)
for idx, row in neutral_df.iterrows():
    neutral_map[make_key(row)].append(idx)
for tidx, trow in target_df.iterrows():
    key = make_key(trow)
    if key in neutral_map:
        for nidx in neutral_map[key]:
            pairs.append((nidx, tidx, "strict"))

if len(pairs) < 100:
    neutral_map2 = defaultdict(list)
    for idx, row in neutral_df.iterrows():
        neutral_map2[make_key(row, include_take=False)].append(idx)
    used = set((n, t) for n, t, _ in pairs)
    for tidx, trow in target_df.iterrows():
        key = make_key(trow, include_take=False)
        if key in neutral_map2:
            for nidx in neutral_map2[key]:
                if (nidx, tidx) not in used:
                    pairs.append((nidx, tidx, "relaxed"))

pairs_df = pd.DataFrame(pairs, columns=["src_idx", "tgt_idx", "pair_type"])
pairs_df["target_emotion"] = [df_work.iloc[t][COL_EMOTION] for t in pairs_df["tgt_idx"]]
pairs_df["speaker"] = [df_work.iloc[s][COL_SPEAKER] for s in pairs_df["src_idx"]]
print(f"\nTotal pairs: {len(pairs_df)}")
print(pairs_df["target_emotion"].value_counts().to_string())

train_pairs, val_pairs = train_test_split(
    pairs_df, test_size=CFG["val_size"], random_state=SEED,
    stratify=pairs_df["target_emotion"] if pairs_df["target_emotion"].nunique() > 1 else None)
train_pairs = train_pairs.reset_index(drop=True)
val_pairs = val_pairs.reset_index(drop=True)

train_indices = set(train_pairs["src_idx"].tolist() + train_pairs["tgt_idx"].tolist())
df_train_rows = df_work.iloc[sorted(train_indices)].reset_index(drop=True)

# Compute global stats
STATS = compute_stats(df_train_rows)
print(f"\nGlobal stats: {STATS}")

# Per-emotion prosody statistics
EMOTION_PROSODY_STATS = {}
for emo in emotion_names:
    emo_df = df_work[df_work[COL_EMOTION] == emo].sample(min(200, len(df_work[df_work[COL_EMOTION] == emo])), random_state=SEED)
    f0_all, e_all = [], []
    for _, row in emo_df.iterrows():
        try:
            feat = load_full_features(row)
            v = feat["f0_hz"] > 0
            if v.any(): f0_all.append(np.log(np.maximum(feat["f0_hz"][v], 1e-6)))
            e_all.append(feat["energy"])
        except: continue
    if f0_all:
        f0c = np.concatenate(f0_all); ec = np.concatenate(e_all)
        EMOTION_PROSODY_STATS[emo] = {
            "f0_log_mean": float(f0c.mean()), "f0_log_std": float(max(f0c.std(), 0.01)),
            "energy_mean": float(ec.mean()), "energy_std": float(max(ec.std(), 0.01)),
            "f0_hz_mean": float(np.exp(f0c.mean()))}
    else:
        EMOTION_PROSODY_STATS[emo] = {"f0_log_mean": STATS["f0_log_mean"],
            "f0_log_std": STATS["f0_log_std"], "energy_mean": STATS["energy_mean"],
            "energy_std": STATS["energy_std"], "f0_hz_mean": 180.0}
    print(f"  {emo:8s}: F0={EMOTION_PROSODY_STATS[emo]['f0_hz_mean']:.0f} Hz")


## 8 · DTW Alignment + Dataset (with F0 Transform)

In [ ]:
# ============================================================
# BLOCK 8 — DTW + EVCPairedDataset (v3: F0 transform + prosody targets)
# ============================================================

def align_1d_by_path(src_len, tgt_1d, wp):
    buckets = [[] for _ in range(src_len)]
    for si, ti in wp:
        if 0 <= si < src_len and 0 <= ti < len(tgt_1d):
            buckets[si].append(tgt_1d[ti])
    out = np.zeros(src_len, dtype=np.float32)
    for i, vals in enumerate(buckets):
        out[i] = float(np.mean(vals)) if vals else tgt_1d[min(max(i, 0), len(tgt_1d)-1)]
    return out

def align_mel_by_dtw(src_mel, tgt_mel, tgt_f0, tgt_e, tgt_v, cache_key=None):
    sT, tT = src_mel.shape[1], tgt_mel.shape[1]
    fallback = lambda: (fit_length_2d(tgt_mel, sT), fit_length_1d(tgt_f0, sT),
                        fit_length_1d(tgt_e, sT), fit_length_1d(tgt_v, sT))
    if sT <= 1 or tT <= 1 or not CFG["use_dtw_alignment"]: return fallback()
    if max(sT, tT) > CFG["max_dtw_frames"]: return fallback()

    if cache_key:
        cp = CACHE_DIR / f"v3_{cache_key}.npz"
        if cp.exists():
            z = np.load(cp)
            if z["mel"].shape[1] == sT: return z["mel"], z["f0"], z["energy"], z["voiced"]

    try:
        X, Y = normalize_mel(src_mel), normalize_mel(tgt_mel)
        _, wp = librosa.sequence.dtw(X=X, Y=Y, metric="cosine")
        wp = wp[::-1]
        aligned_mel = np.zeros((CFG["n_mels"], sT), dtype=np.float32)
        for si in range(sT):
            tis = [int(ti) for s, ti in wp if int(s) == si]
            aligned_mel[:, si] = tgt_mel[:, tis].mean(axis=1) if tis else tgt_mel[:, min(si, tT-1)]
        af0 = align_1d_by_path(sT, tgt_f0, wp)
        ae = align_1d_by_path(sT, tgt_e, wp)
        av = (align_1d_by_path(sT, tgt_v, wp) > 0.5).astype(np.float32)
    except Exception:
        return fallback()

    if cache_key:
        np.savez_compressed(CACHE_DIR / f"v3_{cache_key}.npz",
                            mel=aligned_mel, f0=af0, energy=ae, voiced=av)
    return aligned_mel, af0, ae, av

def log_f0_transform(src_f0_hz, src_emo, tgt_emo):
    weight = CFG.get("f0_transform_weight", 0.5)
    ss = EMOTION_PROSODY_STATS.get(src_emo)
    ts = EMOTION_PROSODY_STATS.get(tgt_emo)
    if ss is None or ts is None: return src_f0_hz.copy()
    f0 = src_f0_hz.copy()
    voiced = f0 > 0
    if not voiced.any(): return f0
    log_f0 = np.log(np.maximum(f0[voiced], 1e-6))
    transformed = (log_f0 - ss["f0_log_mean"]) / (ss["f0_log_std"] + 1e-8) * ts["f0_log_std"] + ts["f0_log_mean"]
    f0[voiced] = np.exp(log_f0 * (1 - weight) + transformed * weight)
    f0[voiced] = np.clip(f0[voiced], 50.0, 600.0)
    return f0

class EVCPairedDataset(Dataset):
    def __init__(self, pairs_df, train=True):
        self.pairs_df = pairs_df.reset_index(drop=True)
        self.train = train
    def __len__(self): return len(self.pairs_df)
    def __getitem__(self, idx):
        item = self.pairs_df.iloc[idx]
        src_row = df_work.iloc[int(item["src_idx"])]
        tgt_row = df_work.iloc[int(item["tgt_idx"])]
        src = load_full_features(src_row); tgt = load_full_features(tgt_row)
        ck = f"pair_{int(item['src_idx'])}_{int(item['tgt_idx'])}"
        tgt_mel_a, tgt_f0_a, tgt_e_a, tgt_v_a = align_mel_by_dtw(
            src["mel_db"], tgt["mel_db"], tgt["f0_hz"], tgt["energy"], tgt["voiced"], cache_key=ck)
        T = src["mel_db"].shape[1]
        src_mel_n = normalize_mel(src["mel_db"]); tgt_mel_n = normalize_mel(tgt_mel_a)
        src_f0_n, src_v = normalize_f0(src["f0_hz"], STATS)
        tgt_f0_n, tgt_v = normalize_f0(tgt_f0_a, STATS)
        src_e_n = normalize_energy(src["energy"], STATS)
        tgt_e_n = normalize_energy(tgt_e_a, STATS)
        src_aux = np.stack([src_f0_n, src_e_n, src_v], axis=0)
        tgt_aux = np.stack([tgt_f0_n, tgt_e_n, tgt_v], axis=0)
        transformed_f0_hz = log_f0_transform(src["f0_hz"], src_row[COL_EMOTION], tgt_row[COL_EMOTION])
        transformed_f0_n, _ = normalize_f0(transformed_f0_hz, STATS)
        tp = EMOTION_PROSODY_STATS.get(tgt_row[COL_EMOTION], EMOTION_PROSODY_STATS["neutral"])
        prosody_cond = np.array([
            (tp["f0_log_mean"] - STATS["f0_log_mean"]) / (STATS["f0_log_std"] + 1e-8),
            tp["f0_log_std"] / (STATS["f0_log_std"] + 1e-8),
            (tp["energy_mean"] - STATS["energy_mean"]) / (STATS["energy_std"] + 1e-8),
            tp["energy_std"] / (STATS["energy_std"] + 1e-8)], dtype=np.float32)
        return {
            "src_mel": torch.from_numpy(src_mel_n), "tgt_mel": torch.from_numpy(tgt_mel_n),
            "src_aux": torch.from_numpy(src_aux), "tgt_aux": torch.from_numpy(tgt_aux),
            "tgt_f0_norm": torch.from_numpy(tgt_f0_n),
            "tgt_energy_norm": torch.from_numpy(tgt_e_n),
            "tgt_voiced": torch.from_numpy(tgt_v),
            "transformed_f0": torch.from_numpy(transformed_f0_n),
            "prosody_cond": torch.from_numpy(prosody_cond),
            "mask": torch.ones(T, dtype=torch.float32),
            "src_emo": torch.tensor(emotion_to_id[src_row[COL_EMOTION]], dtype=torch.long),
            "tgt_emo": torch.tensor(emotion_to_id[tgt_row[COL_EMOTION]], dtype=torch.long),
            "spk_id": torch.tensor(speaker_to_id[src_row[COL_SPEAKER]], dtype=torch.long),
            "pair_index": torch.tensor(idx, dtype=torch.long)}

def collate_fn(batch):
    max_T = max(b["src_mel"].shape[-1] for b in batch)
    out = {}
    for key in batch[0]:
        vals = [b[key] for b in batch]
        if vals[0].dim() == 0: out[key] = torch.stack(vals)
        elif key == "prosody_cond": out[key] = torch.stack(vals)
        elif vals[0].dim() == 1:
            padded = torch.zeros(len(vals), max_T)
            for i, v in enumerate(vals): padded[i, :v.shape[0]] = v
            out[key] = padded
        elif vals[0].dim() == 2:
            C = vals[0].shape[0]
            fill = -1.0 if key in ("src_mel", "tgt_mel") else 0.0
            padded = torch.full((len(vals), C, max_T), fill)
            for i, v in enumerate(vals): padded[i, :, :v.shape[-1]] = v
            out[key] = padded
    return out

train_ds = EVCPairedDataset(train_pairs); val_ds = EVCPairedDataset(val_pairs, train=False)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], collate_fn=collate_fn, pin_memory=True, drop_last=True,
                          persistent_workers=(CFG["num_workers"] > 0))
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], collate_fn=collate_fn, pin_memory=True,
                        persistent_workers=(CFG["num_workers"] > 0))

# SER datasets
class SERDataset(Dataset):
    def __init__(self, df_sub):
        self.data = df_sub.reset_index(drop=True)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        feat = load_full_features(row)
        return torch.from_numpy(normalize_mel(feat["mel_db"])), torch.tensor(emotion_to_id[row[COL_EMOTION]], dtype=torch.long), torch.tensor(idx)

def ser_collate(batch):
    mels, emos, idxs = zip(*batch)
    max_T = max(m.shape[-1] for m in mels)
    padded = torch.full((len(mels), CFG["n_mels"], max_T), -1.0)
    for i, m in enumerate(mels): padded[i, :, :m.shape[-1]] = m
    return padded, torch.stack(emos), torch.stack(idxs)

ser_train_df = df_train_rows[df_train_rows[COL_EMOTION].isin(emotion_names)].reset_index(drop=True)
ser_val_df = df_work[~df_work.index.isin(train_indices)][lambda x: x[COL_EMOTION].isin(emotion_names)].reset_index(drop=True)
ser_train_loader = DataLoader(SERDataset(ser_train_df), batch_size=32, shuffle=True, num_workers=2, collate_fn=ser_collate, drop_last=True)
ser_val_loader = DataLoader(SERDataset(ser_val_df), batch_size=32, shuffle=False, num_workers=2, collate_fn=ser_collate)

print(f"Train: {len(train_ds)} pairs, Val: {len(val_ds)} pairs")
print(f"SER train: {len(ser_train_df)}, SER val: {len(ser_val_df)}")


## 9 · Model Definitions (v3 with ProsodyHead)

In [ ]:
# ============================================================
# BLOCK 9 — Models: Generator w/ ProsodyHead, Discriminator, SER
# ============================================================

class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, k=5, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=k//2),
            nn.InstanceNorm1d(out_ch, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity())
    def forward(self, x): return self.net(x)

class ContentEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        d = CFG["dropout"]
        self.net = nn.Sequential(
            ConvBlock1D(CFG["n_mels"], 128, k=7, dropout=d),
            ConvBlock1D(128, 192, k=5, dropout=d),
            ConvBlock1D(192, CFG["content_dim"], k=5, dropout=d),
            ConvBlock1D(CFG["content_dim"], CFG["content_dim"], k=3, dropout=d))
    def forward(self, mel): return self.net(mel)

class AuxEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        d = CFG["dropout"]
        self.net = nn.Sequential(
            ConvBlock1D(4, 32, k=5, dropout=d),  # 4 channels: f0, energy, voiced, transformed_f0
            ConvBlock1D(32, CFG["aux_dim"], k=5, dropout=d))
    def forward(self, aux): return self.net(aux)

class ProsodyHead(nn.Module):
    def __init__(self):
        super().__init__()
        h = CFG["hidden_dim"]
        self.f0_head = nn.Sequential(ConvBlock1D(h, 128, k=5), ConvBlock1D(128, 64, k=3), nn.Conv1d(64, 1, 1))
        self.energy_head = nn.Sequential(ConvBlock1D(h, 64, k=5), nn.Conv1d(64, 1, 1))
        self.voiced_head = nn.Sequential(ConvBlock1D(h, 64, k=3), nn.Conv1d(64, 1, 1), nn.Sigmoid())
    def forward(self, hidden):
        return self.f0_head(hidden).squeeze(1), self.energy_head(hidden).squeeze(1), self.voiced_head(hidden).squeeze(1)

class Decoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        h = CFG["hidden_dim"]; d = CFG["dropout"]
        self.layers = nn.Sequential(
            ConvBlock1D(in_dim, h, k=5, dropout=d),
            ConvBlock1D(h, h, k=5, dropout=d),
            ConvBlock1D(h, h, k=3, dropout=d))
        self.mel_out = nn.Sequential(nn.Conv1d(h, CFG["n_mels"], 1), nn.Tanh())
    def forward(self, x):
        hidden = self.layers(x)
        return self.mel_out(hidden), hidden

class EVCGenerator(nn.Module):
    def __init__(self, n_speakers, n_emotions):
        super().__init__()
        self.content_encoder = ContentEncoder()
        self.aux_encoder = AuxEncoder()
        self.spk_emb = nn.Embedding(n_speakers, CFG["speaker_dim"])
        self.emo_emb = nn.Embedding(n_emotions, CFG["emotion_dim"])
        self.prosody_proj = nn.Linear(4, CFG["prosody_cond_dim"])
        in_dim = CFG["content_dim"] + CFG["aux_dim"] + CFG["speaker_dim"] + CFG["emotion_dim"] + CFG["prosody_cond_dim"]
        self.decoder = Decoder(in_dim)
        self.prosody_head = ProsodyHead()

    def forward(self, src_mel, src_aux_4ch, spk_id, tgt_emo, prosody_cond=None,
                return_content=False, return_prosody=True):
        B, _, T = src_mel.shape
        content = self.content_encoder(src_mel)
        aux = self.aux_encoder(src_aux_4ch)
        spk = self.spk_emb(spk_id).unsqueeze(-1).expand(-1, -1, T)
        emo = self.emo_emb(tgt_emo).unsqueeze(-1).expand(-1, -1, T)
        if prosody_cond is not None:
            pc = self.prosody_proj(prosody_cond).unsqueeze(-1).expand(-1, -1, T)
        else:
            pc = torch.zeros(B, CFG["prosody_cond_dim"], T, device=src_mel.device)
        x = torch.cat([content, aux, spk, emo, pc], dim=1)
        mel_out, hidden = self.decoder(x)
        if return_prosody:
            f0_p, e_p, v_p = self.prosody_head(hidden)
            if return_content: return mel_out, f0_p, e_p, v_p, content
            return mel_out, f0_p, e_p, v_p
        if return_content: return mel_out, content
        return mel_out

class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha): ctx.alpha = alpha; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return g.neg() * ctx.alpha, None

class EmotionFromContent(nn.Module):
    def __init__(self, n_emotions):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(CFG["content_dim"], 128), nn.ReLU(True), nn.Dropout(0.2), nn.Linear(128, n_emotions))
    def forward(self, content_feat, alpha=1.0):
        pooled = content_feat.mean(dim=2)
        return self.net(_GradReverse.apply(pooled, alpha))

class MelDiscriminator(nn.Module):
    def __init__(self, n_emotions):
        super().__init__()
        self.emo_emb = nn.Embedding(n_emotions, 16)
        self.net = nn.Sequential(
            nn.Conv1d(CFG["n_mels"]+16, 128, 5, padding=2), nn.LeakyReLU(0.2, True),
            nn.Conv1d(128, 128, 5, padding=2), nn.LeakyReLU(0.2, True),
            nn.Conv1d(128, 64, 5, padding=2), nn.LeakyReLU(0.2, True),
            nn.Conv1d(64, 1, 1))
    def forward(self, mel, emo_id):
        B, _, T = mel.shape
        emo = self.emo_emb(emo_id).unsqueeze(-1).expand(-1, -1, T)
        return self.net(torch.cat([mel, emo], dim=1)).mean(dim=[1, 2])

class SERClassifier(nn.Module):
    def __init__(self, n_emotions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(CFG["n_mels"], 96, 5, padding=2), nn.BatchNorm1d(96), nn.ReLU(True), nn.MaxPool1d(2),
            nn.Conv1d(96, 160, 5, padding=2), nn.BatchNorm1d(160), nn.ReLU(True), nn.MaxPool1d(2),
            nn.Conv1d(160, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(True), nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(256, n_emotions)
    def forward(self, mel): return self.fc(self.net(mel).squeeze(-1))

# Instantiate
n_spk = len(speaker_to_id); n_emo = len(emotion_to_id)
G = EVCGenerator(n_spk, n_emo).to(DEVICE)
D = MelDiscriminator(n_emo).to(DEVICE)
SER = SERClassifier(n_emo).to(DEVICE)
EMO_FROM_CONTENT = EmotionFromContent(n_emo).to(DEVICE)
SER_ONLINE = SERClassifier(n_emo).to(DEVICE)

ct = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"G: {ct(G):,}  D: {ct(D):,}  SER: {ct(SER):,}")


## 10 · Checkpoint, SER, Phase Schedule, Training Loop

In [ ]:
# ============================================================
# BLOCK 10 — Checkpoint utilities + SER + Phase + Training
# ============================================================
import re as _re

def _epoch_from_name(p):
    m = _re.search(r"epoch[_-]?(\d+)", p.stem, _re.IGNORECASE)
    return int(m.group(1)) if m else -1

def find_latest_checkpoint():
    candidates = []
    for d in [CHECKPOINT_INPUT_DIR, CKPT_DIR]:
        if d.exists():
            for c in sorted(d.rglob("*.pt")):
                if c.name.startswith("ser_"): continue
                candidates.append(c)
    scored = []
    for c in candidates:
        try:
            head = torch.load(c, map_location="cpu", weights_only=False)
            if not isinstance(head, dict) or "G" not in head: continue
            scored.append((max(_epoch_from_name(c), head.get("epoch", -1)), c))
        except: continue
    return sorted(scored)[-1][1] if scored else None

def save_checkpoint(epoch, tag="latest"):
    ckpt = {"epoch": epoch, "G": G.state_dict(), "D": D.state_dict(), "SER": SER.state_dict(),
            "EMO_FROM_CONTENT": EMO_FROM_CONTENT.state_dict(), "SER_ONLINE": SER_ONLINE.state_dict(),
            "opt_G": opt_G.state_dict(), "opt_D": opt_D.state_dict(),
            "history": history, "emotion_to_id": emotion_to_id, "speaker_to_id": speaker_to_id,
            "stats": STATS, "emotion_prosody_stats": EMOTION_PROSODY_STATS, "cfg": CFG}
    torch.save(ckpt, CKPT_DIR / f"evc_v3_{tag}.pt")

def load_checkpoint(path):
    global history
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    if "G" not in ckpt: return 0
    G.load_state_dict(ckpt["G"], strict=False)
    if "D" in ckpt: D.load_state_dict(ckpt["D"], strict=False)
    if "SER" in ckpt: SER.load_state_dict(ckpt["SER"], strict=False)
    if ckpt.get("EMO_FROM_CONTENT"):
        try: EMO_FROM_CONTENT.load_state_dict(ckpt["EMO_FROM_CONTENT"])
        except: pass
    if ckpt.get("SER_ONLINE"):
        try: SER_ONLINE.load_state_dict(ckpt["SER_ONLINE"])
        except: pass
    if isinstance(ckpt.get("history"), list): history = ckpt["history"]
    return ckpt.get("epoch", 0)

# --- SER pretrain/load ---
ser_loaded = False
latest_ckpt = find_latest_checkpoint()
if latest_ckpt:
    try:
        ck = torch.load(latest_ckpt, map_location=DEVICE, weights_only=False)
        if "SER" in ck: SER.load_state_dict(ck["SER"]); ser_loaded = True; print(f"SER loaded from {latest_ckpt.name}")
    except: pass

if not ser_loaded:
    print("Pretraining SER...")
    opt_ser = torch.optim.AdamW(SER.parameters(), lr=CFG["lr_SER"])
    best_acc = 0
    for ep in range(1, CFG["ser_pretrain_epochs"]+1):
        SER.train()
        for mel, y, _ in ser_train_loader:
            mel, y = mel.to(DEVICE), y.to(DEVICE)
            opt_ser.zero_grad(); loss = F.cross_entropy(SER(mel), y); loss.backward()
            nn.utils.clip_grad_norm_(SER.parameters(), 5.0); opt_ser.step()
        SER.eval(); correct, total = 0, 0
        with torch.no_grad():
            for mel, y, _ in ser_val_loader:
                mel, y = mel.to(DEVICE), y.to(DEVICE)
                correct += (SER(mel).argmax(1) == y).sum().item(); total += y.numel()
        acc = correct / max(1, total)
        if acc > best_acc: best_acc = acc; torch.save(SER.state_dict(), CKPT_DIR / "ser_best.pt")
        if ep % 5 == 0: print(f"  SER ep {ep}: acc={acc:.3f}")
    print(f"SER best: {best_acc:.3f}")

SER.eval()
for p in SER.parameters(): p.requires_grad_(False)
if CFG["use_online_ser"]:
    SER_ONLINE.load_state_dict(SER.state_dict())
    for p in SER_ONLINE.parameters(): p.requires_grad_(True)
print("SER frozen. Online SER ready.")

# --- Optimizers ---
opt_G = torch.optim.AdamW(G.parameters(), lr=CFG["lr_G"], betas=(0.5, 0.9), weight_decay=CFG["weight_decay"])
opt_D = torch.optim.AdamW(D.parameters(), lr=CFG["lr_D"], betas=(0.5, 0.9), weight_decay=CFG["weight_decay"])
opt_grl = torch.optim.AdamW(EMO_FROM_CONTENT.parameters(), lr=CFG["lr_G"], betas=(0.5, 0.9))
opt_ser_online = torch.optim.AdamW(SER_ONLINE.parameters(), lr=CFG["lr_ser_online"], betas=(0.5, 0.9))
history = []
content_teacher = None

# --- Resume ---
start_epoch = 1
if CFG["resume"] and latest_ckpt:
    ep = load_checkpoint(latest_ckpt)
    if ep > 0:
        start_epoch = ep + 1
        print(f"Resumed from epoch {ep}, next = {start_epoch}")
        print("  (ProsodyHead layers initialized fresh)")

if start_epoch > CFG["phase1_epochs"] + 1:
    content_teacher = copy.deepcopy(G.content_encoder).to(DEVICE)
    content_teacher.eval()
    for p in content_teacher.parameters(): p.requires_grad_(False)
    print("Content teacher restored.")

print(f"Starting from epoch {start_epoch}")


## 11 · Phase Schedule + Training Loop

In [ ]:
# ============================================================
# BLOCK 11 — Phase schedule, losses, main training loop
# ============================================================

def lerp(a, b, t): return a + (b - a) * float(np.clip(t, 0.0, 1.0))

def get_phase(epoch):
    if epoch <= CFG["phase1_epochs"]:
        return {"name": "P1-recon+F0warmup", "mode": "reconstruct",
                "l_l1": 20.0, "l_content": 0, "l_cycle": 0, "l_energy": 2.0, "l_ser": 0, "l_adv": 0,
                "l_f0": 4.0, "l_epred": 2.0, "l_voiced": 1.0, "l_prosody": 0, "l_pf0": 0, "l_grl": 0,
                "lr_G_s": 1.0, "lr_D_s": 0, "noise": 0}
    if epoch <= CFG["phase1_epochs"] + CFG["phase2_epochs"]:
        t = (epoch - CFG["phase1_epochs"]) / CFG["phase2_epochs"]
        return {"name": "P2-emotion+F0", "mode": "convert",
                "l_l1": lerp(CFG["p2_lambda_l1"], CFG["p2_lambda_l1"]*0.8, t),
                "l_content": CFG["p2_lambda_content"], "l_cycle": CFG["p2_lambda_cycle"], "l_energy": 2.0,
                "l_ser": lerp(CFG["p2_lambda_ser"]*0.5, CFG["p2_lambda_ser"], t),
                "l_adv": lerp(CFG["p2_lambda_adv"]*0.5, CFG["p2_lambda_adv"], t),
                "l_f0": CFG["lambda_f0"], "l_epred": CFG["lambda_energy_pred"],
                "l_voiced": CFG["lambda_voiced"], "l_prosody": CFG["lambda_prosody"],
                "l_pf0": CFG["lambda_prosody_f0"],
                "l_grl": CFG["lambda_grl"] if CFG["use_grl"] else 0,
                "lr_G_s": 1.0, "lr_D_s": 1.0, "noise": lerp(0.03, 0.01, t)}
    t = (epoch - CFG["phase1_epochs"] - CFG["phase2_epochs"]) / CFG["phase3_epochs"]
    return {"name": "P3-sharpen", "mode": "convert",
            "l_l1": lerp(CFG["p3_lambda_l1"], CFG["p3_lambda_l1"]*0.7, t),
            "l_content": CFG["p3_lambda_content"], "l_cycle": CFG["p3_lambda_cycle"], "l_energy": 2.0,
            "l_ser": lerp(CFG["p3_lambda_ser"]*0.8, CFG["p3_lambda_ser"], t),
            "l_adv": lerp(CFG["p3_lambda_adv"]*0.7, CFG["p3_lambda_adv"], t),
            "l_f0": CFG["lambda_f0"]*1.2, "l_epred": CFG["lambda_energy_pred"],
            "l_voiced": CFG["lambda_voiced"], "l_prosody": CFG["lambda_prosody"]*1.2,
            "l_pf0": CFG["lambda_prosody_f0"]*1.2,
            "l_grl": CFG["lambda_grl"] if CFG["use_grl"] else 0,
            "lr_G_s": lerp(1.0, 0.5, t), "lr_D_s": 1.0, "noise": lerp(0.01, 0, t)}

def set_lr(opt, base, scale):
    for pg in opt.param_groups: pg["lr"] = base * scale

def gate(mel, mask): return mel * mask[:, None, :]

def masked_l1(pred, tgt, mask):
    m = mask[:, None, :]; loss = torch.abs(pred - tgt) * m
    return loss.sum() / (m.sum() * pred.shape[1] + 1e-8)

def masked_l1_1d(pred, tgt, mask):
    return (torch.abs(pred - tgt) * mask).sum() / (mask.sum() + 1e-8)

def f0_sup_loss(f0_pred, tgt_f0, tgt_voiced, mask):
    vm = (tgt_voiced > 0.5).float() * mask
    if vm.sum() < 1: return torch.tensor(0.0, device=f0_pred.device)
    return (torch.abs(f0_pred - tgt_f0) * vm).sum() / (vm.sum() + 1e-8)

def voiced_bce(pred, tgt, mask):
    return F.binary_cross_entropy(pred * mask, tgt * mask, reduction='sum') / (mask.sum() + 1e-8)

def _mm(x, m): return (x * m).sum(1) / (m.sum(1) + 1e-8)
def _ms(x, m):
    mu = _mm(x, m).unsqueeze(1)
    return torch.sqrt((((x - mu)**2) * m).sum(1) / (m.sum(1) + 1e-8) + 1e-8)

def prosody_energy_loss(fake, tgt_aux, mask):
    ge = fake.mean(dim=1); te = tgt_aux[:, 1, :]
    return F.l1_loss(_mm(ge, mask), _mm(te, mask)) + F.l1_loss(_ms(ge, mask), _ms(te, mask))

def f0_stats_loss(f0_pred, tgt_f0, tgt_voiced, mask):
    vm = (tgt_voiced > 0.5).float() * mask
    if vm.sum() < 10: return torch.tensor(0.0, device=f0_pred.device)
    return F.l1_loss(_mm(f0_pred, vm), _mm(tgt_f0, vm)) + F.l1_loss(_ms(f0_pred, vm), _ms(tgt_f0, vm))

# ─── MAIN TRAINING LOOP ─────────────────────────────────────────────────────
for epoch in range(start_epoch, CFG["total_epochs"] + 1):
    ph = get_phase(epoch)
    set_lr(opt_G, CFG["lr_G"], ph["lr_G_s"]); set_lr(opt_D, CFG["lr_D"], ph["lr_D_s"])
    if epoch == CFG["phase1_epochs"] + 1 and content_teacher is None:
        content_teacher = copy.deepcopy(G.content_encoder).to(DEVICE).eval()
        for p in content_teacher.parameters(): p.requires_grad_(False)

    G.train(); D.train(); totals = defaultdict(float); seen = 0; tic = time.time()
    for batch in tqdm(train_loader, desc=f"Ep{epoch:03d} {ph['name']}", leave=False):
        sm = batch["src_mel"].to(DEVICE); tm = batch["tgt_mel"].to(DEVICE)
        mask = batch["mask"].to(DEVICE); se = batch["src_emo"].to(DEVICE); te = batch["tgt_emo"].to(DEVICE)
        spk = batch["spk_id"].to(DEVICE); pc = batch["prosody_cond"].to(DEVICE)
        sa3 = batch["src_aux"].to(DEVICE); tf0 = batch["transformed_f0"].to(DEVICE)
        sa4 = torch.cat([sa3, tf0.unsqueeze(1)], dim=1)
        tgt_f0n = batch["tgt_f0_norm"].to(DEVICE); tgt_en = batch["tgt_energy_norm"].to(DEVICE)
        tgt_v = batch["tgt_voiced"].to(DEVICE); tgt_aux = batch["tgt_aux"].to(DEVICE)

        dm, de = (sm, se) if ph["mode"] == "reconstruct" else (tm, te)
        df0 = sa3[:, 0, :] if ph["mode"] == "reconstruct" else tgt_f0n
        den = sa3[:, 1, :] if ph["mode"] == "reconstruct" else tgt_en
        dv = sa3[:, 2, :] if ph["mode"] == "reconstruct" else tgt_v

        # Discriminator
        lD = torch.tensor(0.0, device=DEVICE)
        if ph["l_adv"] > 0:
            with torch.no_grad(): fd, _, _, _ = G(sm, sa4, spk, de, pc); fd = gate(fd, mask)
            noise = ph["noise"]
            pr = D(dm + torch.randn_like(dm)*noise, de); pf = D(fd + torch.randn_like(fd)*noise, de)
            lD = 0.5*(F.mse_loss(pr, torch.ones_like(pr)) + F.mse_loss(pf, torch.zeros_like(pf)))
            opt_D.zero_grad(set_to_none=True); lD.backward()
            nn.utils.clip_grad_norm_(D.parameters(), CFG["grad_clip"]); opt_D.step()

        # Online SER
        if CFG["use_online_ser"] and ph["mode"] == "convert":
            with torch.no_grad(): fs, _, _, _ = G(sm, sa4, spk, de, pc); fs = gate(fs, mask)
            SER_ONLINE.train()
            lo = F.cross_entropy(SER_ONLINE(dm), de) + F.cross_entropy(SER_ONLINE(fs), de)
            opt_ser_online.zero_grad(set_to_none=True); lo.backward()
            nn.utils.clip_grad_norm_(SER_ONLINE.parameters(), CFG["grad_clip"]); opt_ser_online.step()

        # Generator
        use_grl = CFG["use_grl"] and ph["l_grl"] > 0
        if use_grl:
            fake, f0p, ep, vp, cf = G(sm, sa4, spk, de, pc, return_content=True, return_prosody=True)
        else:
            fake, f0p, ep, vp = G(sm, sa4, spk, de, pc, return_prosody=True)
        fake = gate(fake, mask)

        ll1 = masked_l1(fake, dm, mask)
        le = masked_l1_1d(fake.mean(1), dm.mean(1), mask)
        lf0 = f0_sup_loss(f0p, df0, dv, mask)
        lep = masked_l1_1d(ep, den, mask)
        lv = voiced_bce(vp, dv, mask)

        lc = torch.tensor(0.0, device=DEVICE)
        if content_teacher and ph["l_content"] > 0:
            lc = masked_l1(content_teacher(fake), content_teacher(sm).detach(), mask)
        lcy = torch.tensor(0.0, device=DEVICE)
        if ph["l_cycle"] > 0:
            cyc, _, _, _ = G(fake, sa4, spk, se, pc, return_prosody=True); lcy = masked_l1(gate(cyc, mask), sm, mask)
        ls = torch.tensor(0.0, device=DEVICE)
        if ph["l_ser"] > 0:
            ls = F.cross_entropy(SER(fake), de)
            if CFG["use_online_ser"]: SER_ONLINE.eval(); ls = 0.5*ls + 0.5*F.cross_entropy(SER_ONLINE(fake), de)
        lpro = prosody_energy_loss(fake, tgt_aux, mask) if ph["l_prosody"] > 0 else torch.tensor(0.0, device=DEVICE)
        lpf0 = f0_stats_loss(f0p, df0, dv, mask) if ph["l_pf0"] > 0 else torch.tensor(0.0, device=DEVICE)
        lgrl = F.cross_entropy(EMO_FROM_CONTENT(cf, CFG["grl_alpha"]), se) if use_grl else torch.tensor(0.0, device=DEVICE)
        ladv = F.mse_loss(D(fake, de), torch.ones(fake.size(0), device=DEVICE)) if ph["l_adv"] > 0 else torch.tensor(0.0, device=DEVICE)

        lG = (ph["l_l1"]*ll1 + ph["l_energy"]*le + ph["l_f0"]*lf0 + ph["l_epred"]*lep + ph["l_voiced"]*lv
              + ph["l_content"]*lc + ph["l_cycle"]*lcy + ph["l_ser"]*ls + ph["l_adv"]*ladv
              + ph["l_prosody"]*lpro + ph["l_pf0"]*lpf0 + ph["l_grl"]*lgrl)

        opt_G.zero_grad(set_to_none=True)
        if use_grl: opt_grl.zero_grad(set_to_none=True)
        lG.backward(); nn.utils.clip_grad_norm_(G.parameters(), CFG["grad_clip"]); opt_G.step()
        if use_grl: nn.utils.clip_grad_norm_(EMO_FROM_CONTENT.parameters(), CFG["grad_clip"]); opt_grl.step()

        bs = sm.size(0); seen += bs
        for k, v in {"total": lG, "l1": ll1, "f0": lf0, "ser": ls, "adv": ladv}.items():
            totals[k] += v.item() * bs

    metrics = {k: v/max(1,seen) for k,v in totals.items()}
    elapsed = time.time() - tic
    history.append({"epoch": epoch, "phase": ph["name"], **{f"train_{k}": round(v,5) for k,v in metrics.items()}, "time": round(elapsed,1)})
    pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)

    if epoch % CFG["save_every"] == 0 or epoch == CFG["total_epochs"]:
        save_checkpoint(epoch, tag=f"epoch_{epoch:03d}")
    save_checkpoint(epoch, tag="latest")
    print(f"Ep{epoch:03d} | total={metrics.get('total',0):.3f} f0={metrics.get('f0',0):.4f} l1={metrics.get('l1',0):.3f} ser={metrics.get('ser',0):.3f} | {elapsed:.0f}s")

print("\nTraining complete!")


## 12 · NSF-HiFiGAN Neural Vocoder (mel + F0 -> natural waveform)

Griffin-Lim re-invents pitch from the mel and washes out the converted emotion. **NSF-HiFiGAN** takes the generator's mel **and** the ProsodyHead's predicted F0, using a neural source-filter design so the converted pitch is rendered as natural Bangla speech. It is fine-tuned on raw SUBESCO audio (optionally GTA-fine-tuned on the generator's own mels). Griffin-Lim remains only as an emergency fallback if the vocoder is untrained.

In [ ]:
# ============================================================
# BLOCK 12A — NSF-HiFiGAN config + raw audio acquisition
# ============================================================
# NSF-HiFiGAN is a neural Source-Filter vocoder. It consumes the SAME mel the
# generator produces PLUS an explicit F0 contour, and synthesizes a waveform
# whose pitch follows that F0. This is exactly what lets the converted emotion
# (carried by the ProsodyHead's predicted F0) actually be HEARD -- unlike
# Griffin-Lim, which re-invents pitch from the mel and washes the emotion out.

CFG.update({
    # vocoder: "nsf-hifigan" (neural, F0-controllable). "griffinlim" is the only fallback.
    "vocoder": "nsf-hifigan",
    "train_vocoder": True,            # False = skip training (e.g. when loading a checkpoint)

    # ---- raw audio sources (a neural vocoder must train on real waveforms) ----
    "raw_audio_dir": None,            # local folder of .wav files (highest priority)
    "subesco_kaggle_slug": None,      # optional Kaggle mirror, e.g. "<user>/subesco"
    "subesco_hf_id": "sustcsenlp/bn_emotion_speech_corpus",  # official SUBESCO (CC-BY-4.0)
    "max_voc_utts": 4000,             # cap on utterances used for vocoder training

    # ---- NSF-HiFiGAN architecture (upsample product MUST equal hop_length) ----
    "voc": {
        "harmonic_num": 8,
        "upsample_rates": [8, 8, 4, 2],          # 8*8*4*2 = 512 = hop_length
        "upsample_kernel_sizes": [16, 16, 8, 4],
        "upsample_initial_channel": 512,
        "resblock_kernel_sizes": [3, 7, 11],
        "resblock_dilation_sizes": [[1, 3, 5], [1, 3, 5], [1, 3, 5]],
        "segment_frames": 32,         # 32 * 512 = 16384 samples (~1.02 s) per training segment
        "batch_size": 32,             # A100 40GB fits this comfortably (raise to 48 if free)
        "lr": 2.0e-4,
        "adam_betas": (0.8, 0.99),
        "lr_decay": 0.999,
        "lambda_mel": 45.0,
        "max_steps": 200000,          # more steps = more natural; A100 ~ a few it/s
        "log_every": 200,
        "save_every": 5000,           # periodic safety save (checkpointing is not the focus)
        "amp": True,                  # bf16 autocast on A100 (faster, more VRAM headroom)
        # --- optional GTA fine-tune on the generator's OWN mels (closes synthetic-mel gap) ---
        "gta_finetune": True,
        "gta_steps": 20000,
        "gta_max_utts": 2000,
        "gta_lr": 1.0e-4,
        # --- noise-robustness knobs ---
        "sine_amp": 0.1,              # source excitation amplitude
        "noise_std": 0.003,           # source noise std (lower = less breathy hiss)
        "mel_augment": True,          # perturb mels in training so the vocoder tolerates
        "mel_aug_prob": 0.5,          #   the EVC generator's over-smoothed/predicted mels
    },
    "vocoder_ckpt": None,             # path to a pretrained NSF-HiFiGAN .pt to load instead of training
    # --- generated-audio cleanup (applied by synthesize_waveform) ---
    "denoise": True,
    "denoise_highpass_hz": 55.0,      # remove sub-sonic rumble / DC
    "denoise_gate": True,             # attenuate residual hiss in near-silent frames
    "denoise_gate_db": -45.0,
    "denoise_spectral": False,        # set True if `noisereduce` is installed (extra hiss removal)
})

assert int(np.prod(CFG["voc"]["upsample_rates"])) == CFG["hop_length"], \
    "NSF-HiFiGAN upsample product must equal hop_length (currently %d)" % CFG["hop_length"]

VOC_CKPT_PATH = CKPT_DIR / "nsf_hifigan.pt"

def _ensure_pkg(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except Exception:
        _pip(pip_name or import_name)
        return __import__(import_name)

def acquire_raw_audio():
    """Return a directory containing SUBESCO .wav files, or None.
    Priority: explicit local dir -> Kaggle mirror -> HuggingFace (official CC-BY-4.0)."""
    # 1) explicit local dir
    if CFG["raw_audio_dir"]:
        d = Path(CFG["raw_audio_dir"])
        if d.exists() and list(d.rglob("*.wav")):
            print(f"Raw audio: using local dir {d}")
            return d
    # 2) Kaggle mirror
    if CFG["subesco_kaggle_slug"]:
        dst = Path("/content/subesco_raw"); dst.mkdir(parents=True, exist_ok=True)
        if not list(dst.rglob("*.wav")):
            try:
                subprocess.run(["kaggle", "datasets", "download", "-d",
                                CFG["subesco_kaggle_slug"], "-p", str(dst), "--unzip"], check=True)
            except Exception as e:
                print("Kaggle SUBESCO download failed:", e)
        if list(dst.rglob("*.wav")):
            print(f"Raw audio: using Kaggle mirror at {dst}")
            return dst
    # 3) HuggingFace official corpus
    try:
        dst = Path("/content/subesco_hf")
        if not (dst.exists() and list(dst.rglob("*.wav"))):
            dst.mkdir(parents=True, exist_ok=True)
            _ensure_pkg("datasets")
            from datasets import load_dataset, Audio
            print(f"Downloading SUBESCO from HuggingFace: {CFG['subesco_hf_id']} ...")
            ds = load_dataset(CFG["subesco_hf_id"], split="train")
            acol = None
            for c in ds.column_names:
                if ds.features[c].__class__.__name__ == "Audio":
                    acol = c; break
            acol = acol or "audio"
            ds = ds.cast_column(acol, Audio(sampling_rate=CFG["sample_rate"]))
            n = min(len(ds), CFG["max_voc_utts"])
            for i in range(n):
                ex = ds[i]
                arr = np.asarray(ex[acol]["array"], dtype=np.float32)
                src_path = ex[acol].get("path") or f"utt_{i:05d}.wav"
                stem = Path(src_path).stem
                sf.write(dst / f"{stem}.wav", arr, CFG["sample_rate"])
            print(f"Raw audio: wrote {n} wavs to {dst}")
        return dst
    except Exception as e:
        print("HuggingFace SUBESCO download failed:", e)
    return None

RAW_AUDIO_DIR = None
VOC_SAMPLES = []
if CFG["vocoder"] == "nsf-hifigan":
    RAW_AUDIO_DIR = acquire_raw_audio()

if RAW_AUDIO_DIR is not None:
    raw_wavs = {p.stem.lower(): p for p in RAW_AUDIO_DIR.rglob("*.wav")}
    print(f"Found {len(raw_wavs)} raw .wav files")
    # Try to PAIR each metadata row's precomputed mel/F0 with its raw wav (exact mel match)
    paired = 0
    for _, row in df_work.iterrows():
        stem = Path(str(row[COL_MEL])).stem.lower()
        stem2 = stem[:-4] if stem.endswith("_mel") else stem
        wav = raw_wavs.get(stem) or raw_wavs.get(stem2)
        if wav is not None:
            VOC_SAMPLES.append({"wav": str(wav), "mel": row[COL_MEL],
                                "f0": row[COL_F0], "voiced": row[COL_VOICED],
                                "energy": row[COL_ENERGY], "emotion": row[COL_EMOTION],
                                "speaker": row[COL_SPEAKER]})
            paired += 1
    if paired >= max(50, int(0.05 * len(df_work))):
        print(f"Vocoder data: PAIRED mode -- {paired} (precomputed mel/F0) <-> raw wav (exact mel match)")
    else:
        VOC_SAMPLES = [{"wav": str(p)} for p in list(raw_wavs.values())[:CFG["max_voc_utts"]]]
        print(f"Vocoder data: SELF-EXTRACT mode -- {len(VOC_SAMPLES)} wavs "
              f"(stem pairing matched only {paired}); mel/F0 computed from audio")
else:
    print("No raw audio found -> NSF-HiFiGAN training disabled; using Griffin-Lim fallback.")
    if CFG["vocoder"] == "nsf-hifigan":
        CFG["vocoder"] = "griffinlim"


In [ ]:
# ============================================================
# BLOCK 12B — Vocoder mel / F0 utilities (consistent with the EVC mel space)
# ============================================================
# Mel basis identical to the EVC feature definition (slaney norm, same fmin/fmax/n_mels).
_VOC_MEL_BASIS_NP = librosa.filters.mel(sr=CFG["sample_rate"], n_fft=CFG["n_fft"],
                                        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])

def wav_to_mel_db(wav):
    """Mel in the SAME dB space the EVC pipeline assumes (power_to_db ref=1.0, clipped [-80,0])."""
    S = librosa.feature.melspectrogram(
        y=np.asarray(wav, dtype=np.float32), sr=CFG["sample_rate"], n_fft=CFG["n_fft"],
        hop_length=CFG["hop_length"], win_length=CFG["win_length"], n_mels=CFG["n_mels"],
        fmin=CFG["fmin"], fmax=CFG["fmax"], power=2.0)
    mel_db = librosa.power_to_db(S, ref=1.0)
    return np.clip(mel_db, -80.0, 0.0).astype(np.float32)

def wav_to_f0_hz(wav, n_frames):
    """Frame-level F0 (Hz, 0 for unvoiced) via librosa.pyin, aligned to n_frames."""
    try:
        f0, _, _ = librosa.pyin(np.asarray(wav, dtype=np.float32),
                                fmin=60.0, fmax=600.0, sr=CFG["sample_rate"],
                                frame_length=CFG["win_length"], hop_length=CFG["hop_length"])
        f0 = np.nan_to_num(f0, nan=0.0).astype(np.float32)
    except Exception:
        f0 = np.zeros(n_frames, dtype=np.float32)
    return fit_length_1d(f0, n_frames)

# Differentiable torch log-mel for the reconstruction loss (no torchaudio dependency).
_VOC_MEL_BASIS = torch.from_numpy(_VOC_MEL_BASIS_NP).float().to(DEVICE)
_VOC_HANN = torch.hann_window(CFG["win_length"]).to(DEVICE)

def torch_mel_for_loss(wav):
    """wav: (B, L) -> (B, n_mels, T) log-mel, matched to the EVC mel config."""
    spec = torch.stft(wav, n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
                      win_length=CFG["win_length"], window=_VOC_HANN,
                      center=True, return_complex=True)
    mag = spec.abs() ** 2
    mel = torch.matmul(_VOC_MEL_BASIS, mag)
    return torch.log(torch.clamp(mel, min=1e-5))

from scipy.signal import butter, sosfilt

def postprocess_audio(wav, voiced=None):
    """Light, deterministic cleanup to reduce vocoder hiss/rumble.
    high-pass -> energy noise-gate on near-silent frames -> (optional spectral) -> peak norm."""
    wav = np.asarray(wav, dtype=np.float32)
    if wav.size == 0:
        return wav
    sr = CFG["sample_rate"]
    # 1) high-pass (kill sub-sonic rumble / DC offset that reads as low-freq noise)
    hp = float(CFG.get("denoise_highpass_hz", 0.0))
    if hp and hp > 0:
        sos = butter(4, hp / (sr / 2.0), btype="highpass", output="sos")
        wav = sosfilt(sos, wav).astype(np.float32)
    # 2) frame-wise noise gate: attenuate frames far below the peak (residual hiss in pauses)
    if CFG.get("denoise_gate", True):
        hop = CFG["hop_length"]; n = len(wav); nf = max(1, n // hop)
        frames = wav[:nf * hop].reshape(nf, hop)
        rms = np.sqrt((frames ** 2).mean(axis=1) + 1e-12)
        thr = (rms.max() + 1e-9) * (10.0 ** (float(CFG.get("denoise_gate_db", -45.0)) / 20.0))
        gain = np.where(rms < thr, 0.1, 1.0).astype(np.float32)
        gain = np.convolve(gain, np.ones(3, np.float32) / 3.0, mode="same")  # smooth transitions
        gain_full = np.repeat(gain, hop)
        if len(gain_full) < n:
            gain_full = np.pad(gain_full, (0, n - len(gain_full)), constant_values=float(gain[-1]))
        wav = (wav[:len(gain_full)] * gain_full[:len(wav)]).astype(np.float32)
    # 3) optional spectral denoise (only if the library is present)
    if CFG.get("denoise_spectral", False):
        try:
            import noisereduce as nr
            wav = nr.reduce_noise(y=wav, sr=sr, stationary=True, prop_decrease=0.75).astype(np.float32)
        except Exception:
            pass
    # 4) peak normalize
    m = float(np.abs(wav).max())
    if m > 0:
        wav = wav / (m + 1e-8) * 0.97
    return wav.astype(np.float32)

print("Vocoder mel/F0 utilities ready (with denoise post-processing).")


In [ ]:
# ============================================================
# BLOCK 12C — NSF-HiFiGAN model (Neural Source-Filter + HiFi-GAN)
# ============================================================
from torch.nn.utils import weight_norm, remove_weight_norm

_LRELU = 0.1

def _voc_init_weights(m, mean=0.0, std=0.01):
    if m.__class__.__name__.find("Conv") != -1:
        m.weight.data.normal_(mean, std)

def _get_padding(k, d=1):
    return int((k * d - d) / 2)

class SineGen(nn.Module):
    """Generate a (fundamental + harmonics) sine excitation from an F0 contour."""
    def __init__(self, samp_rate, harmonic_num=8, sine_amp=0.1,
                 noise_std=0.003, voiced_threshold=0.0):
        super().__init__()
        self.sine_amp = sine_amp; self.noise_std = noise_std
        self.harmonic_num = harmonic_num; self.sampling_rate = samp_rate
        self.voiced_threshold = voiced_threshold; self.dim = harmonic_num + 1

    def _f02uv(self, f0):
        return (f0 > self.voiced_threshold).float()

    @torch.no_grad()
    def forward(self, f0):
        # f0: (B, T, 1) in Hz at the audio sample rate
        fn = f0 * torch.arange(1, self.dim + 1, device=f0.device, dtype=f0.dtype).reshape(1, 1, -1)
        rad = fn / float(self.sampling_rate)
        rand_ini = torch.rand(f0.shape[0], 1, self.dim, device=f0.device, dtype=f0.dtype)
        rand_ini[:, :, 0] = 0.0
        phase = torch.cumsum(rad, dim=1) + rand_ini
        sines = torch.sin(2.0 * np.pi * phase)
        uv = self._f02uv(f0)
        noise_amp = uv * self.noise_std + (1.0 - uv) * self.sine_amp / 3.0
        noise = noise_amp * torch.randn_like(sines)
        sine_waves = sines * self.sine_amp * uv + noise
        return sine_waves, uv, noise

class SourceModuleHnNSF(nn.Module):
    def __init__(self, samp_rate, harmonic_num=8, sine_amp=0.1,
                 add_noise_std=0.003, voiced_threshold=0.0):
        super().__init__()
        self.sine_amp = sine_amp
        self.l_sin_gen = SineGen(samp_rate, harmonic_num, sine_amp, add_noise_std, voiced_threshold)
        self.l_linear = nn.Linear(harmonic_num + 1, 1)
        self.l_tanh = nn.Tanh()

    def forward(self, x):
        # x: (B, T_samples, 1) F0 in Hz at sample rate
        sine_wavs, uv, _ = self.l_sin_gen(x)
        sine_merge = self.l_tanh(self.l_linear(sine_wavs))
        return sine_merge, uv, None

class ResBlock1(nn.Module):
    def __init__(self, channels, kernel_size=3, dilation=(1, 3, 5)):
        super().__init__()
        self.convs1 = nn.ModuleList([
            weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=d,
                                  padding=_get_padding(kernel_size, d))) for d in dilation])
        self.convs2 = nn.ModuleList([
            weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=1,
                                  padding=_get_padding(kernel_size, 1))) for _ in dilation])
        self.convs1.apply(_voc_init_weights); self.convs2.apply(_voc_init_weights)

    def forward(self, x):
        for c1, c2 in zip(self.convs1, self.convs2):
            xt = F.leaky_relu(x, _LRELU); xt = c1(xt)
            xt = F.leaky_relu(xt, _LRELU); xt = c2(xt)
            x = xt + x
        return x

    def remove_weight_norm(self):
        for l in self.convs1: remove_weight_norm(l)
        for l in self.convs2: remove_weight_norm(l)

class HiFiGANNSFGenerator(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.h = h
        self.num_kernels = len(h["resblock_kernel_sizes"])
        self.num_upsamples = len(h["upsample_rates"])
        self.upp = int(np.prod(h["upsample_rates"]))
        self.m_source = SourceModuleHnNSF(h["sampling_rate"], h["harmonic_num"],
                                          h.get("sine_amp", 0.1), h.get("noise_std", 0.003))
        self.conv_pre = weight_norm(nn.Conv1d(h["num_mels"], h["upsample_initial_channel"], 7, 1, 3))
        self.ups = nn.ModuleList(); self.noise_convs = nn.ModuleList()
        for i, (u, k) in enumerate(zip(h["upsample_rates"], h["upsample_kernel_sizes"])):
            c_cur = h["upsample_initial_channel"] // (2 ** (i + 1))
            self.ups.append(weight_norm(nn.ConvTranspose1d(
                h["upsample_initial_channel"] // (2 ** i), c_cur, k, u, padding=(k - u) // 2)))
            if i + 1 < self.num_upsamples:
                stride_f0 = int(np.prod(h["upsample_rates"][i + 1:]))
                self.noise_convs.append(nn.Conv1d(
                    1, c_cur, kernel_size=stride_f0 * 2, stride=stride_f0, padding=stride_f0 // 2))
            else:
                self.noise_convs.append(nn.Conv1d(1, c_cur, kernel_size=1))
        self.resblocks = nn.ModuleList(); ch = h["upsample_initial_channel"]
        for i in range(self.num_upsamples):
            ch = h["upsample_initial_channel"] // (2 ** (i + 1))
            for k, d in zip(h["resblock_kernel_sizes"], h["resblock_dilation_sizes"]):
                self.resblocks.append(ResBlock1(ch, k, d))
        self.conv_post = weight_norm(nn.Conv1d(ch, 1, 7, 1, 3))
        self.ups.apply(_voc_init_weights); self.conv_post.apply(_voc_init_weights)

    def forward(self, x, f0):
        # x: (B, n_mels, T)   f0: (B, T) in Hz
        f0 = f0[:, None, :]
        f0_up = F.interpolate(f0, scale_factor=self.upp, mode="nearest").transpose(1, 2)
        har_source, _, _ = self.m_source(f0_up)
        har_source = har_source.transpose(1, 2)        # (B, 1, T*upp)
        x = self.conv_pre(x)
        for i in range(self.num_upsamples):
            x = F.leaky_relu(x, _LRELU)
            x = self.ups[i](x)
            x = x + self.noise_convs[i](har_source)
            xs = None
            for j in range(self.num_kernels):
                r = self.resblocks[i * self.num_kernels + j](x)
                xs = r if xs is None else xs + r
            x = xs / self.num_kernels
        x = F.leaky_relu(x)
        x = torch.tanh(self.conv_post(x))
        return x

    def remove_weight_norm(self):
        for l in self.ups: remove_weight_norm(l)
        for b in self.resblocks: b.remove_weight_norm()
        remove_weight_norm(self.conv_pre); remove_weight_norm(self.conv_post)

class DiscriminatorP(nn.Module):
    def __init__(self, period, kernel_size=5, stride=3):
        super().__init__()
        self.period = period
        self.convs = nn.ModuleList([
            weight_norm(nn.Conv2d(1, 32, (kernel_size, 1), (stride, 1), padding=(_get_padding(kernel_size), 0))),
            weight_norm(nn.Conv2d(32, 128, (kernel_size, 1), (stride, 1), padding=(_get_padding(kernel_size), 0))),
            weight_norm(nn.Conv2d(128, 512, (kernel_size, 1), (stride, 1), padding=(_get_padding(kernel_size), 0))),
            weight_norm(nn.Conv2d(512, 1024, (kernel_size, 1), (stride, 1), padding=(_get_padding(kernel_size), 0))),
            weight_norm(nn.Conv2d(1024, 1024, (kernel_size, 1), 1, padding=(2, 0))),
        ])
        self.conv_post = weight_norm(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, x):
        fmap = []; b, c, t = x.shape
        if t % self.period != 0:
            x = F.pad(x, (0, self.period - (t % self.period)), "reflect"); t = x.shape[-1]
        x = x.view(b, c, t // self.period, self.period)
        for l in self.convs:
            x = F.leaky_relu(l(x), _LRELU); fmap.append(x)
        x = self.conv_post(x); fmap.append(x)
        return torch.flatten(x, 1, -1), fmap

class MultiPeriodDiscriminator(nn.Module):
    def __init__(self, periods=(2, 3, 5, 7, 11)):
        super().__init__()
        self.discriminators = nn.ModuleList([DiscriminatorP(p) for p in periods])
    def forward(self, y, y_hat):
        yr, yg, fr, fg = [], [], [], []
        for d in self.discriminators:
            a, fa = d(y); b, fb = d(y_hat)
            yr.append(a); fr.append(fa); yg.append(b); fg.append(fb)
        return yr, yg, fr, fg

class DiscriminatorS(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.ModuleList([
            weight_norm(nn.Conv1d(1, 128, 15, 1, padding=7)),
            weight_norm(nn.Conv1d(128, 128, 41, 2, groups=4, padding=20)),
            weight_norm(nn.Conv1d(128, 256, 41, 2, groups=16, padding=20)),
            weight_norm(nn.Conv1d(256, 512, 41, 4, groups=16, padding=20)),
            weight_norm(nn.Conv1d(512, 1024, 41, 4, groups=16, padding=20)),
            weight_norm(nn.Conv1d(1024, 1024, 41, 1, groups=16, padding=20)),
            weight_norm(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
        ])
        self.conv_post = weight_norm(nn.Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x):
        fmap = []
        for l in self.convs:
            x = F.leaky_relu(l(x), _LRELU); fmap.append(x)
        x = self.conv_post(x); fmap.append(x)
        return torch.flatten(x, 1, -1), fmap

class MultiScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([DiscriminatorS(), DiscriminatorS(), DiscriminatorS()])
        self.meanpools = nn.ModuleList([nn.AvgPool1d(4, 2, padding=2), nn.AvgPool1d(4, 2, padding=2)])
    def forward(self, y, y_hat):
        yr, yg, fr, fg = [], [], [], []
        for i, d in enumerate(self.discriminators):
            if i != 0:
                y = self.meanpools[i - 1](y); y_hat = self.meanpools[i - 1](y_hat)
            a, fa = d(y); b, fb = d(y_hat)
            yr.append(a); fr.append(fa); yg.append(b); fg.append(fb)
        return yr, yg, fr, fg

def feature_loss(fmap_r, fmap_g):
    loss = 0.0
    for dr, dg in zip(fmap_r, fmap_g):
        for r, g in zip(dr, dg):
            loss = loss + torch.mean(torch.abs(r - g))
    return loss * 2.0

def discriminator_loss(real, gen):
    loss = 0.0
    for dr, dg in zip(real, gen):
        loss = loss + torch.mean((1 - dr) ** 2) + torch.mean(dg ** 2)
    return loss

def generator_loss(gen):
    loss = 0.0
    for dg in gen:
        loss = loss + torch.mean((1 - dg) ** 2)
    return loss

_h = dict(CFG["voc"]); _h.update({"sampling_rate": CFG["sample_rate"], "num_mels": CFG["n_mels"]})
_gtest = HiFiGANNSFGenerator(_h)
print(f"NSF-HiFiGAN generator defined: {sum(p.numel() for p in _gtest.parameters())/1e6:.2f}M params "
      f"| upsample {_h['upsample_rates']} (prod={int(np.prod(_h['upsample_rates']))}=hop)")
del _gtest


In [ ]:
# ============================================================
# BLOCK 12D — Vocoder dataset (mel + F0 + waveform), randomly segmented
# ============================================================
def _voc_smooth_time(m, k):
    """Per-mel-channel temporal moving-average (mimics generator over-smoothing)."""
    pad = k // 2
    mp = np.pad(m, ((0, 0), (pad, pad)), mode="edge")
    ker = np.ones(k, np.float32) / k
    out = np.empty_like(m)
    for i in range(m.shape[0]):
        out[i] = np.convolve(mp[i], ker, mode="valid")[:m.shape[1]]
    return out.astype(np.float32)

def augment_mel(mel):
    """Perturb a normalized mel so the vocoder learns to tolerate the EVC generator's
    over-smoothed / slightly noisy predicted mels -> much less hiss at inference."""
    m = mel.copy()
    if random.random() < 0.7:
        m = _voc_smooth_time(m, random.choice([3, 5]))
    if random.random() < 0.7:
        m = m + np.random.randn(*m.shape).astype(np.float32) * random.uniform(0.01, 0.05)
    if random.random() < 0.5:
        m = m * random.uniform(0.95, 1.05)
    return np.clip(m, -1.0, 1.0).astype(np.float32)

class VocoderDataset(Dataset):
    def __init__(self, samples, segment_frames):
        self.samples = samples
        self.seg = segment_frames
        self.seg_len = segment_frames * CFG["hop_length"]

    def __len__(self):
        return len(self.samples)

    def _load(self, s):
        wav, _ = librosa.load(s["wav"], sr=CFG["sample_rate"], mono=True)
        wav = wav.astype(np.float32)
        peak = float(np.abs(wav).max())
        if peak > 0:
            wav = wav / peak * 0.95
        if "mel" in s:                                   # PAIRED: exact EVC representation
            mel_db = load_mel_db(s["mel"]); T = mel_db.shape[1]
            f0 = load_1d_feature(s["f0"], expected_len=T)
        else:                                            # SELF-EXTRACT from the waveform
            mel_db = wav_to_mel_db(wav); T = mel_db.shape[1]
            f0 = wav_to_f0_hz(wav, T)
        mel_n = normalize_mel(mel_db)                    # [-1, 1], generator output space
        need = T * CFG["hop_length"]
        wav = np.pad(wav, (0, need - len(wav))) if len(wav) < need else wav[:need]
        return mel_n, f0.astype(np.float32), wav

    def __getitem__(self, idx):
        mel_n = f0 = wav = None
        for _ in range(4):
            try:
                mel_n, f0, wav = self._load(self.samples[idx]); break
            except Exception:
                idx = (idx + 1) % len(self.samples)
        if mel_n is None:
            mel_n = np.full((CFG["n_mels"], self.seg), -1.0, np.float32)
            f0 = np.zeros(self.seg, np.float32); wav = np.zeros(self.seg_len, np.float32)
        T = mel_n.shape[1]
        if T <= self.seg:
            pad = self.seg - T
            mel_n = np.pad(mel_n, ((0, 0), (0, pad)), constant_values=-1.0)
            f0 = np.pad(f0, (0, pad))
            if len(wav) < self.seg_len:
                wav = np.pad(wav, (0, self.seg_len - len(wav)))
            sfr = 0
        else:
            sfr = random.randint(0, T - self.seg)
        mel_c = mel_n[:, sfr:sfr + self.seg]
        f0_c = f0[sfr:sfr + self.seg]
        wav_c = wav[sfr * CFG["hop_length"]: sfr * CFG["hop_length"] + self.seg_len]
        if len(wav_c) < self.seg_len:
            wav_c = np.pad(wav_c, (0, self.seg_len - len(wav_c)))
        if CFG["voc"].get("mel_augment") and random.random() < CFG["voc"].get("mel_aug_prob", 0.5):
            mel_c = augment_mel(mel_c)
        return (torch.from_numpy(mel_c.copy()), torch.from_numpy(f0_c.copy()),
                torch.from_numpy(wav_c.copy()))

def voc_collate(batch):
    mels, f0s, wavs = zip(*batch)
    return torch.stack(mels), torch.stack(f0s), torch.stack(wavs)

print(f"VocoderDataset ready ({len(VOC_SAMPLES)} samples, "
      f"segment={CFG['voc']['segment_frames']} frames).")


In [ ]:
# ============================================================
# BLOCK 12E — Train NSF-HiFiGAN (A100 bf16-AMP; max-naturalness, save = safety only)
# ============================================================
_VOC_READY = False
voc_G = voc_mpd = voc_msd = None

def build_vocoder():
    h = dict(CFG["voc"]); h.update({"sampling_rate": CFG["sample_rate"], "num_mels": CFG["n_mels"]})
    return HiFiGANNSFGenerator(h).to(DEVICE)

def save_vocoder(step):
    try:
        torch.save({"step": step, "G": voc_G.state_dict(),
                    "mpd": voc_mpd.state_dict() if voc_mpd is not None else None,
                    "msd": voc_msd.state_dict() if voc_msd is not None else None,
                    "cfg_voc": CFG["voc"]}, VOC_CKPT_PATH)
    except Exception as e:
        print("  (checkpoint save skipped:", e, ")")

_VOC_AMP = bool(CFG["voc"].get("amp", True)) and DEVICE.type == "cuda"
_VOC_AMP_DTYPE = torch.bfloat16  # A100-native, stable for GAN training (no GradScaler needed)

if CFG["vocoder"] == "nsf-hifigan":
    voc_G = build_vocoder()
    # Optional: load an explicit pretrained checkpoint and skip training
    if CFG["vocoder_ckpt"] and Path(CFG["vocoder_ckpt"]).exists():
        ck = torch.load(CFG["vocoder_ckpt"], map_location=DEVICE)
        voc_G.load_state_dict(ck["G"], strict=False); _VOC_READY = True
        print(f"Loaded NSF-HiFiGAN from {CFG['vocoder_ckpt']}")

    if CFG["train_vocoder"] and len(VOC_SAMPLES) > 0 and not _VOC_READY:
        vh = CFG["voc"]
        voc_mpd = MultiPeriodDiscriminator().to(DEVICE)
        voc_msd = MultiScaleDiscriminator().to(DEVICE)
        opt_g = torch.optim.AdamW(voc_G.parameters(), vh["lr"], betas=tuple(vh["adam_betas"]))
        opt_d = torch.optim.AdamW(list(voc_mpd.parameters()) + list(voc_msd.parameters()),
                                  vh["lr"], betas=tuple(vh["adam_betas"]))
        sch_g = torch.optim.lr_scheduler.ExponentialLR(opt_g, vh["lr_decay"])
        sch_d = torch.optim.lr_scheduler.ExponentialLR(opt_d, vh["lr_decay"])
        vds = VocoderDataset(VOC_SAMPLES, vh["segment_frames"])
        vdl = DataLoader(vds, batch_size=vh["batch_size"], shuffle=True,
                         num_workers=CFG["num_workers"], collate_fn=voc_collate,
                         pin_memory=True, drop_last=True,
                         persistent_workers=(CFG["num_workers"] > 0),
                         prefetch_factor=(4 if CFG["num_workers"] > 0 else None))
        print(f"Training NSF-HiFiGAN: {len(vds)} utts, {len(vdl)} batches/epoch, "
              f"-> {vh['max_steps']} steps | AMP(bf16)={_VOC_AMP} | batch {vh['batch_size']}")
        voc_G.train(); voc_mpd.train(); voc_msd.train()
        step = 0; t0 = time.time(); done = False
        while not done:
            for mel, f0, wav in vdl:
                mel = mel.to(DEVICE, non_blocking=True); f0 = f0.to(DEVICE, non_blocking=True)
                y = wav.to(DEVICE, non_blocking=True).unsqueeze(1)
                with torch.autocast(device_type="cuda", dtype=_VOC_AMP_DTYPE, enabled=_VOC_AMP):
                    y_g = voc_G(mel, f0)
                # ---- discriminator step ----
                opt_d.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda", dtype=_VOC_AMP_DTYPE, enabled=_VOC_AMP):
                    ydr, ydg, _, _ = voc_mpd(y, y_g.detach())
                    sdr, sdg, _, _ = voc_msd(y, y_g.detach())
                    loss_d = discriminator_loss(ydr, ydg) + discriminator_loss(sdr, sdg)
                loss_d.backward(); opt_d.step()
                # ---- generator step ----
                opt_g.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda", dtype=_VOC_AMP_DTYPE, enabled=_VOC_AMP):
                    ydr, ydg, frs, fgs = voc_mpd(y, y_g)
                    sdr, sdg, sfrs, sfgs = voc_msd(y, y_g)
                    loss_fm = feature_loss(frs, fgs) + feature_loss(sfrs, sfgs)
                    loss_adv = generator_loss(ydg) + generator_loss(sdg)
                loss_mel = F.l1_loss(torch_mel_for_loss(y_g.float().squeeze(1)),
                                     torch_mel_for_loss(y.float().squeeze(1))) * vh["lambda_mel"]
                loss_g = loss_adv + loss_fm + loss_mel
                loss_g.backward(); opt_g.step()
                step += 1
                if step % vh["log_every"] == 0:
                    sps = step / max(1e-9, time.time() - t0)
                    print(f"  voc {step:6d} | mel {loss_mel.item():.3f} fm {loss_fm.item():.2f} "
                          f"adv {loss_adv.item():.3f} d {loss_d.item():.3f} | {sps:.1f} it/s")
                if step % vh["save_every"] == 0:
                    save_vocoder(step)
                if step >= vh["max_steps"]:
                    done = True; break
            sch_g.step(); sch_d.step()
        save_vocoder(step); _VOC_READY = True
        print("NSF-HiFiGAN base training complete.")
    elif not _VOC_READY:
        print("NSF-HiFiGAN not trained and no checkpoint -> falling back to Griffin-Lim.")
        CFG["vocoder"] = "griffinlim"

    if _VOC_READY:
        voc_G.eval()
        for p in voc_G.parameters():
            p.requires_grad_(False)


In [ ]:
# ============================================================
# BLOCK 12F — Vocoder inference wrappers + dispatcher
# ============================================================
@torch.no_grad()
def nsf_hifigan_synthesize(mel_norm, f0_hz, voiced=None):
    """mel_norm: (n_mels, T) in [-1,1]; f0_hz: (T,) Hz (0 = unvoiced) -> waveform (float32)."""
    mel = torch.from_numpy(np.asarray(mel_norm, np.float32)).unsqueeze(0).to(DEVICE)
    f0 = np.asarray(f0_hz, np.float32).copy()
    if voiced is not None:
        v = np.asarray(voiced).reshape(-1)
        n = min(len(v), len(f0)); f0[:n][v[:n] <= 0.5] = 0.0
    f0 = torch.from_numpy(f0).unsqueeze(0).to(DEVICE)
    T = mel.shape[-1]
    if f0.shape[-1] != T:
        f0 = F.interpolate(f0.unsqueeze(1), size=T, mode="nearest").squeeze(1)
    wav = voc_G(mel, f0).squeeze().detach().cpu().numpy().astype(np.float32)
    m = float(np.abs(wav).max())
    if m > 0:
        wav = wav / (m + 1e-8) * 0.95
    return wav

def synthesize_waveform(mel_norm, mel_db, f0_hz, voiced):
    """NSF-HiFiGAN vocoder (Griffin-Lim only as emergency fallback), then denoise cleanup."""
    wav = None
    if CFG.get("vocoder") == "nsf-hifigan" and _VOC_READY:
        try:
            wav = nsf_hifigan_synthesize(mel_norm, f0_hz, voiced)
        except Exception as e:
            print("NSF-HiFiGAN synth failed -> Griffin-Lim:", e)
    if wav is None:
        wav = mel_to_audio_griffinlim(mel_db)
    if CFG.get("denoise", False):
        try:
            wav = postprocess_audio(wav, voiced)
        except Exception as e:
            print("denoise skipped:", e)
    return wav

# self-test (neural path only; world/griffin-lim are defined in the next section)
if _VOC_READY:
    _tT = 40
    _tmel = np.clip(np.random.randn(CFG["n_mels"], _tT) * 0.3 - 0.2, -1, 1).astype(np.float32)
    _tf0 = np.zeros(_tT, np.float32); _tf0[5:35] = 200.0
    _w = nsf_hifigan_synthesize(_tmel, _tf0)
    print(f"NSF-HiFiGAN self-test OK -> {_w.shape[0]} samples "
          f"({_w.shape[0]/CFG['sample_rate']:.2f}s, "
          f"expected {_tT*CFG['hop_length']/CFG['sample_rate']:.2f}s)")
else:
    print(f"Vocoder dispatcher ready (active vocoder: {CFG.get('vocoder')}).")


In [ ]:
# ============================================================
# BLOCK 12G — Optional GTA fine-tune: vocoder on the generator's OWN mels
# ============================================================
# Closes the synthetic-mel gap: the vocoder is fine-tuned on the EVC generator's
# reconstructed mels (paired with the real waveform), so at inference it sees the
# exact (over-smoothed) mel distribution it must vocode -> markedly less hiss.

def _prosody_cond_for(emo):
    tp = EMOTION_PROSODY_STATS.get(emo, EMOTION_PROSODY_STATS.get("neutral"))
    return np.array([
        (tp["f0_log_mean"] - STATS["f0_log_mean"]) / (STATS["f0_log_std"] + 1e-8),
        tp["f0_log_std"] / (STATS["f0_log_std"] + 1e-8),
        (tp["energy_mean"] - STATS["energy_mean"]) / (STATS["energy_std"] + 1e-8),
        tp["energy_std"] / (STATS["energy_std"] + 1e-8)], dtype=np.float32)

class GTADataset(Dataset):
    def __init__(self, items, segment_frames):
        self.items = items; self.seg = segment_frames
        self.seg_len = segment_frames * CFG["hop_length"]
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        it = self.items[idx]
        mel_n = it["gen_mel"]; f0 = it["f0"]
        wav, _ = librosa.load(it["wav"], sr=CFG["sample_rate"], mono=True); wav = wav.astype(np.float32)
        peak = float(np.abs(wav).max())
        if peak > 0: wav = wav / peak * 0.95
        T = mel_n.shape[1]; need = T * CFG["hop_length"]
        wav = np.pad(wav, (0, need - len(wav))) if len(wav) < need else wav[:need]
        if T <= self.seg:
            pad = self.seg - T
            mel_n = np.pad(mel_n, ((0, 0), (0, pad)), constant_values=-1.0)
            f0 = np.pad(f0, (0, pad))
            if len(wav) < self.seg_len: wav = np.pad(wav, (0, self.seg_len - len(wav)))
            sfr = 0
        else:
            sfr = random.randint(0, T - self.seg)
        mel_c = mel_n[:, sfr:sfr + self.seg]; f0_c = f0[sfr:sfr + self.seg]
        wav_c = wav[sfr * CFG["hop_length"]: sfr * CFG["hop_length"] + self.seg_len]
        if len(wav_c) < self.seg_len: wav_c = np.pad(wav_c, (0, self.seg_len - len(wav_c)))
        return (torch.from_numpy(mel_c.copy()), torch.from_numpy(f0_c.astype(np.float32).copy()),
                torch.from_numpy(wav_c.copy()))

GTA_PAIRED = [s for s in VOC_SAMPLES if ("mel" in s and "emotion" in s and "speaker" in s)]
if (CFG.get("vocoder") == "nsf-hifigan" and _VOC_READY and CFG["voc"].get("gta_finetune", True)
        and len(GTA_PAIRED) > 0 and "G" in globals()):
    try:
        vh = CFG["voc"]
        n_use = min(len(GTA_PAIRED), vh["gta_max_utts"])
        print(f"GTA: reconstructing generated mels from {n_use} utterances ...")
        G.eval(); gta_items = []
        for s in GTA_PAIRED[:n_use]:
            try:
                mel_db = load_mel_db(s["mel"]); T = mel_db.shape[1]
                mel_n = normalize_mel(mel_db)
                f0_hz = load_1d_feature(s["f0"], expected_len=T)
                energy = load_1d_feature(s["energy"], expected_len=T)
                f0_n, v = normalize_f0(f0_hz, STATS)
                e_n = normalize_energy(energy, STATS)
                src_aux = np.stack([f0_n, e_n, v], axis=0)
                sa4 = np.concatenate([src_aux, f0_n[None]], axis=0).astype(np.float32)
                sm = torch.from_numpy(mel_n)[None].to(DEVICE)
                sa4_t = torch.from_numpy(sa4)[None].to(DEVICE)
                spk_t = torch.tensor([speaker_to_id[s["speaker"]]], dtype=torch.long, device=DEVICE)
                emo_t = torch.tensor([emotion_to_id[s["emotion"]]], dtype=torch.long, device=DEVICE)
                pc_t = torch.from_numpy(_prosody_cond_for(s["emotion"]))[None].to(DEVICE)
                with torch.no_grad():
                    out = G(sm, sa4_t, spk_t, emo_t, pc_t, return_prosody=True)
                fake_n = out[0][0].detach().cpu().numpy()[:, :T].astype(np.float32)
                gta_items.append({"gen_mel": fake_n, "f0": f0_hz.astype(np.float32), "wav": s["wav"]})
            except Exception:
                continue
        print(f"GTA: built {len(gta_items)} (generated-mel, real-wav) pairs")
        if len(gta_items) >= 32:
            if voc_mpd is None: voc_mpd = MultiPeriodDiscriminator().to(DEVICE)
            if voc_msd is None: voc_msd = MultiScaleDiscriminator().to(DEVICE)
            for p in voc_G.parameters(): p.requires_grad_(True)
            voc_G.train(); voc_mpd.train(); voc_msd.train()
            gta_lr = vh.get("gta_lr", 1e-4)
            opt_g = torch.optim.AdamW(voc_G.parameters(), gta_lr, betas=tuple(vh["adam_betas"]))
            opt_d = torch.optim.AdamW(list(voc_mpd.parameters()) + list(voc_msd.parameters()),
                                      gta_lr, betas=tuple(vh["adam_betas"]))
            gds = GTADataset(gta_items, vh["segment_frames"])
            gdl = DataLoader(gds, batch_size=vh["batch_size"], shuffle=True,
                             num_workers=CFG["num_workers"], collate_fn=voc_collate,
                             pin_memory=True, drop_last=True,
                             persistent_workers=(CFG["num_workers"] > 0),
                         prefetch_factor=(4 if CFG["num_workers"] > 0 else None))
            steps = vh.get("gta_steps", 15000)
            print(f"GTA fine-tune: {len(gds)} pairs, {len(gdl)} batches/epoch, {steps} steps @ lr {gta_lr}")
            step = 0; t0 = time.time(); done = False
            while not done:
                for mel, f0, wav in gdl:
                    mel = mel.to(DEVICE, non_blocking=True); f0 = f0.to(DEVICE, non_blocking=True)
                    y = wav.to(DEVICE, non_blocking=True).unsqueeze(1)
                    with torch.autocast(device_type="cuda", dtype=_VOC_AMP_DTYPE, enabled=_VOC_AMP):
                        y_g = voc_G(mel, f0)
                    opt_d.zero_grad(set_to_none=True)
                    with torch.autocast(device_type="cuda", dtype=_VOC_AMP_DTYPE, enabled=_VOC_AMP):
                        ydr, ydg, _, _ = voc_mpd(y, y_g.detach())
                        sdr, sdg, _, _ = voc_msd(y, y_g.detach())
                        loss_d = discriminator_loss(ydr, ydg) + discriminator_loss(sdr, sdg)
                    loss_d.backward(); opt_d.step()
                    opt_g.zero_grad(set_to_none=True)
                    with torch.autocast(device_type="cuda", dtype=_VOC_AMP_DTYPE, enabled=_VOC_AMP):
                        ydr, ydg, frs, fgs = voc_mpd(y, y_g)
                        sdr, sdg, sfrs, sfgs = voc_msd(y, y_g)
                        loss_fm = feature_loss(frs, fgs) + feature_loss(sfrs, sfgs)
                        loss_adv = generator_loss(ydg) + generator_loss(sdg)
                    loss_mel = F.l1_loss(torch_mel_for_loss(y_g.float().squeeze(1)),
                                         torch_mel_for_loss(y.float().squeeze(1))) * vh["lambda_mel"]
                    loss_g = loss_adv + loss_fm + loss_mel
                    loss_g.backward(); opt_g.step()
                    step += 1
                    if step % vh["log_every"] == 0:
                        print(f"  gta {step:6d} | mel {loss_mel.item():.3f} fm {loss_fm.item():.2f} "
                              f"adv {loss_adv.item():.3f} d {loss_d.item():.3f}")
                    if step >= steps:
                        done = True; break
            save_vocoder(step)
            voc_G.eval()
            for p in voc_G.parameters(): p.requires_grad_(False)
            print("GTA fine-tune complete (vocoder adapted to the generated-mel distribution).")
        else:
            print("GTA: too few pairs built; skipping fine-tune.")
    except Exception as e:
        print("GTA fine-tune skipped due to:", repr(e))
else:
    print("GTA fine-tune not run (disabled, no paired data, or EVC generator/vocoder unavailable).")


## 13 · Inference (NSF-HiFiGAN; Griffin-Lim only as emergency fallback)

`generate_from_pair` runs the generator, denormalizes the mel and the predicted F0, and renders audio through `synthesize_waveform`, which uses the trained NSF-HiFiGAN (mel + predicted F0) and falls back to Griffin-Lim only if the vocoder is unavailable, then applies the `postprocess_audio` denoiser. WORLD has been removed.

In [ ]:
# ============================================================
# BLOCK 12 — Inference (NSF-HiFiGAN vocoder; Griffin-Lim emergency fallback)
# ============================================================
def mel_to_audio_griffinlim(mel_db):
    '''Fallback vocoder.'''
    mel_power = librosa.db_to_power(mel_db, ref=1.0)
    wav = librosa.feature.inverse.mel_to_audio(
        M=mel_power, sr=CFG["sample_rate"], n_fft=CFG["n_fft"],
        hop_length=CFG["hop_length"], win_length=CFG["win_length"],
        fmin=CFG["fmin"], fmax=CFG["fmax"], power=2.0, n_iter=64)
    wav = wav.astype(np.float32)
    if np.abs(wav).max() > 0: wav = wav / (np.abs(wav).max() + 1e-8) * 0.95
    return wav

def load_real_wav(row):
    if COL_WAV is None: return None
    p = resolve_path(row[COL_WAV])
    if p is None: return None
    try:
        wav, _ = librosa.load(str(p), sr=CFG["sample_rate"], mono=True)
        return wav.astype(np.float32)
    except: return None

def estimate_f0_from_wav(wav):
    try:
        f0, _, _ = librosa.pyin(wav, fmin=50, fmax=500, sr=CFG["sample_rate"],
                                frame_length=CFG["win_length"], hop_length=CFG["hop_length"])
        return np.nan_to_num(f0, nan=0.0).astype(np.float32)
    except: return np.zeros(1, dtype=np.float32)

@torch.no_grad()
def generate_from_pair(ds, idx):
    G.eval()
    b = ds[idx]
    sm = b["src_mel"].unsqueeze(0).to(DEVICE)
    sa3 = b["src_aux"].unsqueeze(0).to(DEVICE)
    tf0 = b["transformed_f0"].unsqueeze(0).unsqueeze(1).to(DEVICE)
    sa4 = torch.cat([sa3, tf0], dim=1)
    spk = b["spk_id"].view(1).to(DEVICE)
    te = b["tgt_emo"].view(1).to(DEVICE)
    pc = b["prosody_cond"].unsqueeze(0).to(DEVICE)

    fake_n, f0p, ep, vp = G(sm, sa4, spk, te, pc, return_prosody=True)
    fake_n = fake_n[0].cpu().numpy()
    f0p = f0p[0].cpu().numpy()
    vp = vp[0].cpu().numpy()

    T = b["src_mel"].shape[-1]
    src_db = denormalize_mel(b["src_mel"].numpy())[:, :T]
    tgt_db = denormalize_mel(b["tgt_mel"].numpy())[:, :T]
    gen_db = denormalize_mel(fake_n)[:, :T]

    gen_f0_hz = denormalize_f0(f0p[:T], vp[:T], STATS)
    src_aux_np = b["src_aux"].numpy()[:, :T]
    tgt_aux_np = b["tgt_aux"].numpy()[:, :T]
    src_f0 = denormalize_f0(src_aux_np[0], src_aux_np[2], STATS)
    tgt_f0 = denormalize_f0(tgt_aux_np[0], tgt_aux_np[2], STATS)

    pr = ds.pairs_df.iloc[int(b["pair_index"])]
    src_row = df_work.iloc[int(pr["src_idx"])]; tgt_row = df_work.iloc[int(pr["tgt_idx"])]
    # Normalized mels (vocoder input space = generator Tanh output)
    src_mel_n = b["src_mel"].numpy()[:, :T]
    tgt_mel_n = b["tgt_mel"].numpy()[:, :T]
    gen_mel_n = fake_n[:, :T]

    # Source/target reference: real wav if available, else render via the same vocoder
    src_real = load_real_wav(src_row)
    tgt_real = load_real_wav(tgt_row)
    src_wav = src_real if src_real is not None else synthesize_waveform(src_mel_n, src_db, src_f0, src_aux_np[2])
    tgt_wav = tgt_real if tgt_real is not None else synthesize_waveform(tgt_mel_n, tgt_db, tgt_f0, tgt_aux_np[2])

    # Generated audio: NSF-HiFiGAN (mel + predicted F0), Griffin-Lim emergency fallback
    gen_wav = synthesize_waveform(gen_mel_n, gen_db, gen_f0_hz, vp[:T])

    gen_f0_wav = estimate_f0_from_wav(gen_wav)

    return {"src_db": src_db, "tgt_db": tgt_db, "gen_db": gen_db,
            "src_f0": src_f0, "tgt_f0": tgt_f0, "gen_f0_pred": gen_f0_hz,
            "gen_f0_wav": gen_f0_wav,
            "src_energy": derive_energy_from_mel_db(src_db),
            "tgt_energy": derive_energy_from_mel_db(tgt_db),
            "gen_energy": derive_energy_from_mel_db(gen_db),
            "src_wav": src_wav, "tgt_wav": tgt_wav, "gen_wav": gen_wav,
            "src_emotion": id_to_emotion[int(b["src_emo"])],
            "tgt_emotion": id_to_emotion[int(b["tgt_emo"])],
            "speaker": id_to_speaker[int(b["spk_id"])]}

print("Inference pipeline ready.")


## 13 · Visualization + Audio Playback + Evaluation

In [ ]:
# ============================================================
# BLOCK 13 — Plots + Audio + Evaluation
# ============================================================

def compare(idx=0, save_prefix=None):
    out = generate_from_pair(val_ds, idx)
    t = f"{out['speaker']} | {out['src_emotion']} -> {out['tgt_emotion']}"

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, db, label in zip(axes, [out["src_db"], out["tgt_db"], out["gen_db"]],
                              ["Source (neutral)", f"Target ({out['tgt_emotion']})", f"Generated ({out['tgt_emotion']})"]):
        librosa.display.specshow(db, sr=CFG["sample_rate"], hop_length=CFG["hop_length"],
                                 x_axis="time", y_axis="mel", fmax=CFG["fmax"], ax=ax, cmap="magma")
        ax.set_title(label)
    fig.suptitle(t); plt.tight_layout()
    if save_prefix: fig.savefig(PLOT_DIR / f"{save_prefix}_mel.png", dpi=150, bbox_inches="tight")
    plt.show()

    # F0 comparison (THE diagnostic plot)
    plt.figure(figsize=(14, 4))
    ts = np.arange(len(out["src_f0"])) * CFG["hop_length"] / CFG["sample_rate"]
    tt = np.arange(len(out["tgt_f0"])) * CFG["hop_length"] / CFG["sample_rate"]
    tg = np.arange(len(out["gen_f0_pred"])) * CFG["hop_length"] / CFG["sample_rate"]
    plt.plot(ts, out["src_f0"], 'b-', alpha=0.5, label="Source (neutral)")
    plt.plot(tt, out["tgt_f0"], 'r-', alpha=0.7, label=f"Target ({out['tgt_emotion']})")
    plt.plot(tg, out["gen_f0_pred"], 'g-', lw=2, label="Generated (predicted F0)")
    plt.title("F0 — Green should track RED, not BLUE"); plt.xlabel("Time (s)"); plt.ylabel("Hz")
    plt.legend(); plt.grid(True, alpha=0.3)
    if save_prefix: plt.savefig(PLOT_DIR / f"{save_prefix}_f0.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Audio
    for label, wav in [("SOURCE", out["src_wav"]), ("TARGET", out["tgt_wav"]), ("GENERATED", out["gen_wav"])]:
        print(f"  {label}:")
        display(Audio(wav, rate=CFG["sample_rate"]))
    if save_prefix:
        sf.write(AUDIO_DIR / f"{save_prefix}_source.wav", out["src_wav"], CFG["sample_rate"])
        sf.write(AUDIO_DIR / f"{save_prefix}_target.wav", out["tgt_wav"], CFG["sample_rate"])
        sf.write(AUDIO_DIR / f"{save_prefix}_generated.wav", out["gen_wav"], CFG["sample_rate"])
    return out

# Run evaluation
for i in range(min(5, len(val_ds))):
    print(f"\n{'='*50} Sample {i} {'='*50}")
    compare(i, save_prefix=f"val_{i:03d}")

# Batch evaluation
print("\n\n" + "="*60 + "\n  BATCH EVALUATION\n" + "="*60)
results = []
for i in tqdm(range(min(60, len(val_ds))), desc="Evaluating"):
    out = generate_from_pair(val_ds, i)
    sf0 = out["src_f0"][out["src_f0"]>0]; tf0 = out["tgt_f0"][out["tgt_f0"]>0]
    gf0 = out["gen_f0_pred"][out["gen_f0_pred"]>0]
    sm = float(sf0.mean()) if len(sf0) > 0 else 0
    tm = float(tf0.mean()) if len(tf0) > 0 else 0
    gm = float(gf0.mean()) if len(gf0) > 0 else 0
    moved_f0 = int(abs(gm - tm) < abs(sm - tm))
    results.append({"idx": i, "tgt_emotion": out["tgt_emotion"], "src_f0": round(sm,1),
                    "gen_f0": round(gm,1), "tgt_f0": round(tm,1), "moved_f0": moved_f0})

eval_df = pd.DataFrame(results)
eval_df.to_csv(OUT_DIR / "honest_evaluation_v3.csv", index=False)
print(f"\nF0 moved toward target: {eval_df['moved_f0'].mean():.3f}")
print("\nPer emotion:")
print(eval_df.groupby("tgt_emotion")[["moved_f0", "src_f0", "gen_f0", "tgt_f0"]].mean().round(1))


## 14 · Export Results to Drive

In [ ]:
# ============================================================
# BLOCK 14 — Export
# ============================================================

print(f"All outputs saved to: {OUT_DIR}")
print(f"\nContents:")
for f in sorted(OUT_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(OUT_DIR)} ({f.stat().st_size/1024:.1f} KB)")

# Also create a zip in Drive for easy download
zip_path = DRIVE_ROOT / "evc_v3_results.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", OUT_DIR)
print(f"\nZIP: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)")
print("\nDone! Check Google Drive -> MyDrive -> EVC_Output/")
